In [234]:

# SEAG qualification period is 22 Oct 2024 to 5 Sept 2025

#1. Segregate into Male and Femal 
#2. For each gender perform the following: 
#a. Sort data by mapped eent, then perf scalar (higher the better)
#b. Identify tiers based on performance - Tier 1 is meets bronze medal mark for SEAG, Tier 2 is 2% and Tier 3 is 3.5%
#c. Check - if athlete met bronze or 2%/3.5% then delta_benchmark is zero or +, delta2% is + and delta 3.5% is +
#d. Top ranked athletes for each event are chosen. Max number of athletes for each event is 3, except for 100m/400m which is 6
#    This includes athletes on spex scholarship and potential
#e. The max for each tier is 2. Lower ranked athletes move down one tier.
#3. If athlete qualifies for more than one event the higher tier event is given
#4. Jump and throws junior program to be solved separately

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [235]:
# Import usual modules
import pandas as pd
import csv
import math
import os
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import openpyxl
import datetime
from scipy.stats import lognorm
import re
import string
from bs4 import BeautifulSoup
import requests
import unicodedata # for removing accented characters
import datetime
import icecream as ic
import dateutil.parser as parser 
import datacompy
import pytz
import gspread

from google.cloud import storage



In [236]:
# PRODUCTION ENVIRONMENT
# Extract timed event records

import pandas_gbq
from google.oauth2 import service_account

credentials = service_account.Credentials.from_service_account_file(
    '/Users/veesheenyuen/Desktop/DataScience/Keys/saa-analytics-7c8937b70609.json',
    
    
)

sql1="""
SELECT NAME, RESULT, TEAM, AGE, RANK AS COMPETITION_RANK, DIVISION, EVENT, DISTANCE, EVENT_CLASS, UNIQUE_ID, DOB, NATIONALITY, WIND, CATEGORY_EVENT, GENDER, COMPETITION, DATE, YEAR, REGION, TIMESTAMP
FROM `saa-analytics.results.PRODUCTION_CORRECT` 
WHERE RESULT!='NM' AND RESULT!='-' AND RESULT!='DNS' AND RESULT!='DNF' AND RESULT!='DNQ' AND RESULT!='DQ' AND RESULT IS NOT NULL

"""

competitors = pandas_gbq.read_gbq(sql1, project_id="saa-analytics", credentials=credentials)




Downloading: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████|


In [6]:
competitors

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,DOB,NATIONALITY,WIND,CATEGORY_EVENT,GENDER,COMPETITION,DATE,YEAR,REGION,TIMESTAMP
0,SI EN TABITHA NG,00:11:03.950000,,,4.0,,3000m,,,,,SGP,,Long,Female,14TH ASEAN SCHOOLS GAMES,2025-11-23 00:00:00+00:00,2025,International,2025-12-09 12:38:00+00:00
1,JE AN GARRETT CHUA,6.84,,,3.0,,Long Jump,,,,,SGP,1.0,Jump,Male,14TH ASEAN SCHOOLS GAMES,2025-11-23 00:00:00+00:00,2025,International,2025-12-09 12:38:00+00:00
2,LAUREL JIA EN LIM,27.71,,,6.0,,Discus Throw,,,,,SGP,,Throw,Female,14TH ASEAN SCHOOLS GAMES,2025-11-23 00:00:00+00:00,2025,International,2025-12-09 12:38:00+00:00
3,JOSHUA SHYEN LEE,11.03,,,4.0,,100m,,,,,SGP,-0.5,Sprint,Male,14TH ASEAN SCHOOLS GAMES,2025-11-23 00:00:00+00:00,2025,International,2025-12-09 12:38:00+00:00
4,DARYEN XIN TZE KO,54.65,,,2.0,,400m Hurdles,,,,,SGP,,Hurdles,Male,14TH ASEAN SCHOOLS GAMES,2025-11-23 00:00:00+00:00,2025,International,2025-12-09 12:38:00+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
170101,"YONG HUI, LAI",12.79,ACTIVESG ATHLETICS,34,nan,nan,Triple Jump,,,,1989,,0,Jump,Male,Club Zoom,2023-11-25 00:00:00+00:00,2023,Local,2025-12-07 17:55:00+00:00
170102,"CHEN, KE YUAN",11.72,Nanyang Technological Uni,23,nan,nan,Triple Jump,,,,2000,,0,Jump,Male,SA - 5th All Comers,2023-12-09 00:00:00+00:00,2023,nan,2025-12-07 17:55:00+00:00
170103,"YEOH, YUE QI",8.39,Singapore Management Uni,19,nan,nan,Triple Jump,,,,2004,,0,Jump,Female,SA - 5th All Comers,2023-12-09 00:00:00+00:00,2023,nan,2025-12-07 17:55:00+00:00
170104,"TAN, TSE TENG",10.9,Nanyang Technological Uni,21,nan,nan,Triple Jump,,,,2002,,0,Jump,Female,SA - 5th All Comers,2023-12-09 00:00:00+00:00,2023,nan,2025-12-07 17:55:00+00:00


In [7]:
def convert_time_format(time_str):
    """
    Convert time from 'HH:MM:SS.mmmmmm' to 'MM:SS.mm' format.
    
    Args:
        time_str: Time string in format 'HH:MM:SS.mmmmmm'
    
    Returns:
        Converted time string in format 'MM:SS.mm'
    """
    if pd.isna(time_str):
        return time_str
    
    # Match pattern HH:MM:SS.mmmmmm (with flexible microseconds)
    pattern = r'^(\d{2}):(\d{2}):(\d{2})\.(\d+)$'
    match = re.match(pattern, str(time_str))
    
    if match:
        hours, minutes, seconds, microseconds = match.groups()
        # Take only first 2 digits of microseconds (centiseconds)
        centiseconds = microseconds[:2].ljust(2, '0')
        return f"{minutes}:{seconds}.{centiseconds}"
    
    # Return original if pattern doesn't match
    return time_str


# Apply conversion
competitors['RESULT'] = competitors['RESULT'].apply(convert_time_format)


In [8]:
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/SEAG_u18/')


competitors.to_csv('database_download.csv', sep=',', encoding='utf-8-sig', index=False)

In [9]:
competitors

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,DOB,NATIONALITY,WIND,CATEGORY_EVENT,GENDER,COMPETITION,DATE,YEAR,REGION,TIMESTAMP
0,SI EN TABITHA NG,11:03.95,,,4.0,,3000m,,,,,SGP,,Long,Female,14TH ASEAN SCHOOLS GAMES,2025-11-23 00:00:00+00:00,2025,International,2025-12-09 12:38:00+00:00
1,JE AN GARRETT CHUA,6.84,,,3.0,,Long Jump,,,,,SGP,1.0,Jump,Male,14TH ASEAN SCHOOLS GAMES,2025-11-23 00:00:00+00:00,2025,International,2025-12-09 12:38:00+00:00
2,LAUREL JIA EN LIM,27.71,,,6.0,,Discus Throw,,,,,SGP,,Throw,Female,14TH ASEAN SCHOOLS GAMES,2025-11-23 00:00:00+00:00,2025,International,2025-12-09 12:38:00+00:00
3,JOSHUA SHYEN LEE,11.03,,,4.0,,100m,,,,,SGP,-0.5,Sprint,Male,14TH ASEAN SCHOOLS GAMES,2025-11-23 00:00:00+00:00,2025,International,2025-12-09 12:38:00+00:00
4,DARYEN XIN TZE KO,54.65,,,2.0,,400m Hurdles,,,,,SGP,,Hurdles,Male,14TH ASEAN SCHOOLS GAMES,2025-11-23 00:00:00+00:00,2025,International,2025-12-09 12:38:00+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
170101,"YONG HUI, LAI",12.79,ACTIVESG ATHLETICS,34,nan,nan,Triple Jump,,,,1989,,0,Jump,Male,Club Zoom,2023-11-25 00:00:00+00:00,2023,Local,2025-12-07 17:55:00+00:00
170102,"CHEN, KE YUAN",11.72,Nanyang Technological Uni,23,nan,nan,Triple Jump,,,,2000,,0,Jump,Male,SA - 5th All Comers,2023-12-09 00:00:00+00:00,2023,nan,2025-12-07 17:55:00+00:00
170103,"YEOH, YUE QI",8.39,Singapore Management Uni,19,nan,nan,Triple Jump,,,,2004,,0,Jump,Female,SA - 5th All Comers,2023-12-09 00:00:00+00:00,2023,nan,2025-12-07 17:55:00+00:00
170104,"TAN, TSE TENG",10.9,Nanyang Technological Uni,21,nan,nan,Triple Jump,,,,2002,,0,Jump,Female,SA - 5th All Comers,2023-12-09 00:00:00+00:00,2023,nan,2025-12-07 17:55:00+00:00


In [10]:
# DATE column to contain timezone - tz aware mode

competitors['DATE'] = pd.to_datetime(competitors['DATE'], format='mixed', dayfirst=False, utc=True)


In [11]:
# datetime to contain UTC (timezone)

competitors['NOW'] = datetime.datetime.now()

timezone = pytz.timezone('UTC')

competitors['NOW'] = datetime.datetime.now().replace(tzinfo=timezone)

In [12]:
# Calculate number of days from today to event date

#competitors['DATE'] = pd.to_datetime(competitors['DATE'], format='mixed', dayfirst=False, utc=False)

competitors['delta_time'] = competitors['NOW'] - competitors['DATE']


#competitors['delta_time'] = datetime.datetime.now() - competitors['DATE']


competitors['delta_time_conv'] = pd.to_numeric(competitors['delta_time'].dt.days, downcast='integer')

competitors['event_month'] = competitors['DATE'].dt.month

# Make sure date conversion is is valid for all rows

assert not competitors['delta_time'].isna().any()

In [13]:
competitors[competitors['COMPETITION']=='The 7th Tokai Sprint Games']

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,GENDER,COMPETITION,DATE,YEAR,REGION,TIMESTAMP,NOW,delta_time,delta_time_conv,event_month
2793,Ryan Praharsh,10.70,,,8,,100m,,,,...,Male,The 7th Tokai Sprint Games,2025-08-17 00:00:00+00:00,2025,International,2025-11-17 16:42:00+00:00,2025-12-22 16:43:34.679093+00:00,127 days 16:43:34.679093,127,8
85191,Ryan Praharsh,10.70,,,5,,100m,,,,...,Male,The 7th Tokai Sprint Games,2025-08-17 00:00:00+00:00,2025,International,2025-11-17 16:42:00+00:00,2025-12-22 16:43:34.679093+00:00,127 days 16:43:34.679093,127,8


In [14]:
# Choose date range for SEAG qualification window from Oct 22 to current


#competitors = competitors[(competitors['delta_time_conv']>=0) & (competitors['delta_time_conv']<=365)]

#competitors=competitors.reset_index(drop=True)

competitors['DATE']=competitors['DATE'].dt.tz_localize(None)  # switch off timezone for compatibility with np.datetime64


start = datetime.datetime(2024, 10, 22)
#start = datetime.datetime(2025, 5, 1)


end = datetime.datetime(2025, 9, 14)

start_date = np.datetime64(start)
end_date = np.datetime64(end)


mask = (competitors['DATE'] >= start_date) & (competitors['DATE'] <= end_date)
athletes_selected = competitors.loc[mask]



In [15]:
end_date - start_date

numpy.timedelta64(28252800000000,'us')

In [16]:
athletes_selected[athletes_selected['COMPETITION']=='The 7th Tokai Sprint Games']

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,GENDER,COMPETITION,DATE,YEAR,REGION,TIMESTAMP,NOW,delta_time,delta_time_conv,event_month
2793,Ryan Praharsh,10.70,,,8,,100m,,,,...,Male,The 7th Tokai Sprint Games,2025-08-17,2025,International,2025-11-17 16:42:00+00:00,2025-12-22 16:43:34.679093+00:00,127 days 16:43:34.679093,127,8
85191,Ryan Praharsh,10.70,,,5,,100m,,,,...,Male,The 7th Tokai Sprint Games,2025-08-17,2025,International,2025-11-17 16:42:00+00:00,2025-12-22 16:43:34.679093+00:00,127 days 16:43:34.679093,127,8


In [17]:
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/SEAG_u18/')


athletes_selected.to_csv('athletes_downloaded_Dec21.csv', encoding='utf-8')

In [18]:
# Select all of 2024/25

athletes_selected = competitors[(competitors['YEAR']=='2024')|(competitors['YEAR']=='2025')]

athletes_selected = competitors[(competitors['YEAR']=='2025')]

In [19]:
athletes_selected

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,GENDER,COMPETITION,DATE,YEAR,REGION,TIMESTAMP,NOW,delta_time,delta_time_conv,event_month
0,SI EN TABITHA NG,11:03.95,,,4.0,,3000m,,,,...,Female,14TH ASEAN SCHOOLS GAMES,2025-11-23,2025,International,2025-12-09 12:38:00+00:00,2025-12-22 16:43:34.679093+00:00,29 days 16:43:34.679093,29,11
1,JE AN GARRETT CHUA,6.84,,,3.0,,Long Jump,,,,...,Male,14TH ASEAN SCHOOLS GAMES,2025-11-23,2025,International,2025-12-09 12:38:00+00:00,2025-12-22 16:43:34.679093+00:00,29 days 16:43:34.679093,29,11
2,LAUREL JIA EN LIM,27.71,,,6.0,,Discus Throw,,,,...,Female,14TH ASEAN SCHOOLS GAMES,2025-11-23,2025,International,2025-12-09 12:38:00+00:00,2025-12-22 16:43:34.679093+00:00,29 days 16:43:34.679093,29,11
3,JOSHUA SHYEN LEE,11.03,,,4.0,,100m,,,,...,Male,14TH ASEAN SCHOOLS GAMES,2025-11-23,2025,International,2025-12-09 12:38:00+00:00,2025-12-22 16:43:34.679093+00:00,29 days 16:43:34.679093,29,11
4,DARYEN XIN TZE KO,54.65,,,2.0,,400m Hurdles,,,,...,Male,14TH ASEAN SCHOOLS GAMES,2025-11-23,2025,International,2025-12-09 12:38:00+00:00,2025-12-22 16:43:34.679093+00:00,29 days 16:43:34.679093,29,11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88908,"Ng, Zavier",9.88m,Hwa Chong Institution,14,18,U15,Triple Jump,0.0,None,Z854G11,...,Male,SA Allcomers Meet 2,2025-03-02,2025,Local,2025-11-15 13:47:37.773700,2025-12-22 16:43:34.679093+00:00,295 days 16:43:34.679093,295,3
88909,"Choo Jia Yi, Allyson",9.30m,Team Start Singapore,17.0,3,Open,Triple Jump,0.0,None,A967D08,...,Female,SA Allcomers Meet 3,2025-08-30,2025,Local,2025-11-15 13:49:23.923169,2025-12-22 16:43:34.679093+00:00,114 days 16:43:34.679093,114,8
88914,Lua Yu Xuan,10.02,NYGH,,1.0,C,Triple Jump,,,,...,Female,National School Games,2025-04-18,2025,Local,2025-11-17 17:34:00+00:00,2025-12-22 16:43:34.679093+00:00,248 days 16:43:34.679093,248,4
88915,Muhammad Aaryan Shah Bin Azhar,12.61,SSP,,3.0,B,Triple Jump,,,,...,Male,National School Games,2025-04-13,2025,Local,2025-11-17 17:34:00+00:00,2025-12-22 16:43:34.679093+00:00,253 days 16:43:34.679093,253,4


In [20]:
athletes_selected[athletes_selected['COMPETITION']=='The 7th Tokai Sprint Games']

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,GENDER,COMPETITION,DATE,YEAR,REGION,TIMESTAMP,NOW,delta_time,delta_time_conv,event_month
2793,Ryan Praharsh,10.70,,,8,,100m,,,,...,Male,The 7th Tokai Sprint Games,2025-08-17,2025,International,2025-11-17 16:42:00+00:00,2025-12-22 16:43:34.679093+00:00,127 days 16:43:34.679093,127,8
85191,Ryan Praharsh,10.70,,,5,,100m,,,,...,Male,The 7th Tokai Sprint Games,2025-08-17,2025,International,2025-11-17 16:42:00+00:00,2025-12-22 16:43:34.679093+00:00,127 days 16:43:34.679093,127,8


In [105]:
# Choose 2024/25 only

athletes = athletes_selected

In [106]:
athletes

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,COMPETITION,DATE,YEAR,REGION,TIMESTAMP,NOW,delta_time,delta_time_conv,event_month,clean_name
0,SI EN TABITHA NG,11:03.95,,,4.0,,3000m,,,,...,14TH ASEAN SCHOOLS GAMES,2025-11-23,2025,International,2025-12-09 12:38:00+00:00,2025-12-22 16:43:34.679093+00:00,29 days 16:43:34.679093,29,11,si en tabitha ng
1,JE AN GARRETT CHUA,6.84,,,3.0,,Long Jump,,,,...,14TH ASEAN SCHOOLS GAMES,2025-11-23,2025,International,2025-12-09 12:38:00+00:00,2025-12-22 16:43:34.679093+00:00,29 days 16:43:34.679093,29,11,je an garrett chua
2,LAUREL JIA EN LIM,27.71,,,6.0,,Discus Throw,,,,...,14TH ASEAN SCHOOLS GAMES,2025-11-23,2025,International,2025-12-09 12:38:00+00:00,2025-12-22 16:43:34.679093+00:00,29 days 16:43:34.679093,29,11,laurel jia en lim
3,JOSHUA SHYEN LEE,11.03,,,4.0,,100m,,,,...,14TH ASEAN SCHOOLS GAMES,2025-11-23,2025,International,2025-12-09 12:38:00+00:00,2025-12-22 16:43:34.679093+00:00,29 days 16:43:34.679093,29,11,joshua shyen lee
4,DARYEN XIN TZE KO,54.65,,,2.0,,400m Hurdles,,,,...,14TH ASEAN SCHOOLS GAMES,2025-11-23,2025,International,2025-12-09 12:38:00+00:00,2025-12-22 16:43:34.679093+00:00,29 days 16:43:34.679093,29,11,daryen xin tze ko
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88908,"Ng, Zavier",9.88m,Hwa Chong Institution,14,18,U15,Triple Jump,0.0,None,Z854G11,...,SA Allcomers Meet 2,2025-03-02,2025,Local,2025-11-15 13:47:37.773700,2025-12-22 16:43:34.679093+00:00,295 days 16:43:34.679093,295,3,ng zavier
88909,"Choo Jia Yi, Allyson",9.30m,Team Start Singapore,17.0,3,Open,Triple Jump,0.0,None,A967D08,...,SA Allcomers Meet 3,2025-08-30,2025,Local,2025-11-15 13:49:23.923169,2025-12-22 16:43:34.679093+00:00,114 days 16:43:34.679093,114,8,choo jia yi allyson
88914,Lua Yu Xuan,10.02,NYGH,,1.0,C,Triple Jump,,,,...,National School Games,2025-04-18,2025,Local,2025-11-17 17:34:00+00:00,2025-12-22 16:43:34.679093+00:00,248 days 16:43:34.679093,248,4,lua yu xuan
88915,Muhammad Aaryan Shah Bin Azhar,12.61,SSP,,3.0,B,Triple Jump,,,,...,National School Games,2025-04-13,2025,Local,2025-11-17 17:34:00+00:00,2025-12-22 16:43:34.679093+00:00,253 days 16:43:34.679093,253,4,muhammad aaryan shah bin azhar


## Create dictionary of names and dob

In [107]:
# Strip out punctuation from NAME
from itertools import permutations
import string
from dateutil import parser



translator = str.maketrans('', '', string.punctuation)
athletes['clean_name'] = athletes['NAME'].str.translate(translator)
athletes['clean_name'] = athletes['clean_name'].str.casefold()


athletes = athletes.reset_index(drop=True)

translator = str.maketrans('', '', string.punctuation)

athletes['clean_name'] = athletes['NAME'].apply(lambda x: str(x).translate(translator))
athletes['clean_name'] = athletes['clean_name'].str.casefold()




/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_69858/2990674287.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  athletes['clean_name'] = athletes['NAME'].str.translate(translator)
/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_69858/2990674287.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  athletes['clean_name'] = athletes['clean_name'].str.casefold()


In [108]:
# Function to safely parse DOB

def parse_dob(x):
    if pd.isna(x) or str(x).strip().lower() in ["none", "nan", "nat", ""]:
        return None
    try:
        # force dayfirst since your DOBs are in DD/MM/YY
        dt = parser.parse(str(x), dayfirst=True)
        # fix 2-digit year misinterpretation (like 1912 instead of 2012)
        if dt.year < 1950:  
            dt = dt.replace(year=dt.year + 100)
        return dt.strftime("%Y-%m-%d")
    except Exception:
        return None

athletes["DOB_parsed"] = athletes["DOB"].apply(parse_dob)

# Build dictionary without null DOBs
dictionary_dob_clean_name = {
    row["clean_name"]: row["DOB_parsed"]
    for _, row in athletes.iterrows()
    if row["DOB_parsed"] is not None
}

# Test Aarika Ray
print(dictionary_dob_clean_name.get("aarika ray"))

2012-05-19


In [109]:
dictionary_dob_clean_name

{'rui yong soh': '1991-06-08',
 'shaun goh': '1997-12-01',
 'marc brian louis': '2002-08-07',
 'tate tan fung': '2005-03-02',
 'shanti veronica pereira': '1996-09-20',
 'elizabethann tan': '2003-09-23',
 'kerstin ong jing rong': '1997-06-08',
 'chen xiang ang': '1994-03-07',
 'oliver lim': '2000-07-02',
 'amir rusyaidi osman': '2002-02-19',
 'harry irfan curra': '2008-12-22',
 'vanessa lee ying zhuang': '1998-02-23',
 'reuben lee siong en': '2002-09-17',
 'subaraghav hari': '2006-01-07',
 'jun jie calvin quek': '1996-02-26',
 'nicole low': '1998-02-23',
 'rajan thiruben thana': '2000-10-28',
 'zubin percy muncherji': '1996-06-23',
 'eric yee chun wai': '1999-01-03',
 'jasmine teo': '1992-07-28',
 'keane ko': '2000-07-26',
 'kampton kam': '2001-06-03',
 'tung hon andrew pak': '2002-04-10',
 'jade chew': '2004-01-14',
 'yen young amelia goh': '2008-09-01',
 'andrew george medina': '2002-03-22',
 'feng han lin': '2004-08-03',
 'tia louise rozario': '2000-10-14',
 'chloe chee enya': '2008-

In [110]:
def length(string):

    B = ''
    year = ''

    try:

        length = len(string)

        if length == 2:

            string = '19' + string

        elif length == 1:

            string = ''

        else:

            pass

        if string is not None or len(string) != 1:

            B = parser.parse(string, dayfirst=True)
                        
    except:

        pass

    return B




In [111]:
# Process dates to extract age

# U18 end of 31 Dec 2025 below 18 (>=16 to <18)
# U20 end of 31 Dec 2025 below 20 (>=16 to <20)
# Senior(OPEN) end of 31 Dec 2025 (>=16 yrs)

# Map NSG divisions into age
# Div A = 17 to 20 yrs


mask = (athletes['DIVISION'].str.contains(r'A', na=False) & athletes['COMPETITION'].str.contains(r'National School Games', na=False))
athletes.loc[mask, 'AGE'] = '17.5'

mask = (athletes['DIVISION'].str.contains(r'B', na=False) & athletes['COMPETITION'].str.contains(r'National School Games', na=False))
athletes.loc[mask, 'AGE'] = '15.5'

mask = (athletes['DIVISION'].str.contains(r'C', na=False) & athletes['COMPETITION'].str.contains(r'National School Games', na=False))
athletes.loc[mask, 'AGE'] = '13.5'

mask = (athletes['DIVISION'].str.contains(r'O', na=False) & athletes['COMPETITION'].str.contains(r'National School Games', na=False))  # deprecate
athletes.loc[mask, 'AGE'] = '12'




In [112]:
# Get list of names with no DOB

mask = ((athletes['DOB']=='None')|(athletes['DOB']=='nan'))

#mask = ((final_df['DOB'].isnull()))
#mask = final_df['DOB_new']=='NaT'

missing_names = athletes.loc[mask]

name_list = missing_names['NAME'].str.casefold().to_list()


In [113]:
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/SEAG_u18/')

athletes.to_csv('check.csv', encoding='utf-8')

## Map Events

In [114]:
athletes.columns

Index(['NAME', 'RESULT', 'TEAM', 'AGE', 'COMPETITION_RANK', 'DIVISION',
       'EVENT', 'DISTANCE', 'EVENT_CLASS', 'UNIQUE_ID', 'DOB', 'NATIONALITY',
       'WIND', 'CATEGORY_EVENT', 'GENDER', 'COMPETITION', 'DATE', 'YEAR',
       'REGION', 'TIMESTAMP', 'NOW', 'delta_time', 'delta_time_conv',
       'event_month', 'clean_name', 'DOB_parsed'],
      dtype='object')

In [115]:
# SIMPLE RULES

def simple_map_events(athletes: pd.DataFrame) -> pd.DataFrame:
    # Columns we care about
    str_cols = ['EVENT', 'DISTANCE']
    existing_cols = [c for c in str_cols if c in athletes.columns]

    # Clean text columns
    regex_cleanup = re.compile(r'[\xa0\r\n]|[\x00-\x1f\x7f-\x9f]')
    for col in existing_cols:
        athletes[col] = (
            athletes[col]
            .astype(str)
            .str.replace(regex_cleanup, ' ', regex=True)
            .str.strip()
        )

    # Initialize mapped column
    if 'MAPPED_EVENT' not in athletes.columns:
        athletes['MAPPED_EVENT'] = np.nan

   
    # ----------------------
    # EVENT-only rules (regex on EVENT)
    # ----------------------
    event_rules = {
        r'(Dash|Run).*\b60\b|60 Meter Run|^60m$': '60m',
        r'(Dash|Run).*\b80\b|80 Meter Run|^80m$': '80m',
        r'(Dash|Run).*\b100\b(?!0)|100 Meter Run\b|^100m$': '100m',
        r'(Dash|Run).*\b200\b|^200m$|200\sMeter': '200m',
        r'(Dash|Run).*\b300\b|^300m$|300\sMeter': '300m',
        r'(Dash|Run).*\b400\b|^400m$|400\sMeter': '400m',
        r'(Run.*800|800 Meter Run|^800m$)': '800m',
        r'(Run.*1000|1000m)\b': '1000m',
        r'(Run.*1500|^1500m$)': '1500m',
        r'(Run.*1600|^1600m$)': '1600m',
        r'(Run.*3000|^3000m$)': '3000m',
        r'(Run.*5000|^5000m$)': '5000m',
        r'(Run.*10,000|Run.*10000|^10,000m$|^10000m$|10km|10 km|10,000 m)': '10,000m',
        r'(Run.*Mile|Mile Run|^Mile$|^1 Mile$)': '1 Mile',  # Enhanced Mile mapping

        # NEW: Half Marathon
        r'Half\s*Marathon|21\.0975\s*km|21\s*km': 'Half Marathon', 
        
        # NEW: Marathon
        r'Marathon|42\.195\s*km|42\s*km': 'Marathon',

        # RELAY RULES
        r'4\s*[xX]\s*100m|4x100\s*Relay|400m\s*Relay': '4 x 100m',
        r'4\s*[xX]\s*400m|4x400\s*Relay|1600m\s*Relay': '4 x 400m',
        r'4\s*[xX]\s*200m|4x200\s*Relay|800m\s*Relay': '4 x 200m',
        }

    for pattern, mapped in event_rules.items():
        athletes.loc[athletes['EVENT'].str.contains(pattern, na=False, case=False), 'MAPPED_EVENT'] = mapped

    # ----------------------
    # EVENT + DISTANCE rules
    # ----------------------
    distance_rules = [
        # Short sprints
        {"conditions": {"EVENT": r'(Dash|Run)', "DISTANCE": r'\b60\b'}, "map_to": "60m"},
        {"conditions": {"EVENT": r'(Dash|Run)', "DISTANCE": r'\b80\b'}, "map_to": "80m"},
        {"conditions": {"EVENT": r'(Dash|Run)', "DISTANCE": r'\b100\b'}, "map_to": "100m"},
        {"conditions": {"EVENT": r'(Dash|Run)', "DISTANCE": r'\b150\b'}, "map_to": "150m"},   
        {"conditions": {"EVENT": r'(Dash|Run)', "DISTANCE": r'\b200\b'}, "map_to": "200m"},
        {"conditions": {"EVENT": r'(Dash|Run)', "DISTANCE": r'\b300\b'}, "map_to": "300m"},
        {"conditions": {"EVENT": r'(Dash|Run)', "DISTANCE": r'\b400\b'}, "map_to": "400m"},
        {"conditions": {"EVENT": r'(Dash|Run)', "DISTANCE": r'\b800\b'}, "map_to": "800m"},
        
        # Middle/long
        {"conditions": {"EVENT": r'Run', "DISTANCE": r'\b1500\b'}, "map_to": "1500m"},
        {"conditions": {"EVENT": r'Run', "DISTANCE": r'\b1600\b'}, "map_to": "1600m"},
        {"conditions": {"EVENT": r'Run', "DISTANCE": r'\b2400\b'}, "map_to": "2400m"},
        {"conditions": {"EVENT": r'Run', "DISTANCE": r'\b3000\b'}, "map_to": "3000m"},
        {"conditions": {"EVENT": r'Run', "DISTANCE": r'\b5000\b'}, "map_to": "5000m"},
        {"conditions": {"EVENT": r'Run', "DISTANCE": r'10000'}, "map_to": "10,000m"},
        {"conditions": {"EVENT": r'10000m'}, "map_to": "10,000m"},

        # Road

        {"conditions": {"EVENT": r'5km, Road'}, "map_to": "5km, Road"},
        
        # Mile - NEW
        {"conditions": {"EVENT": r'(Run|Mile)', "DISTANCE": r'(Mile|1609|1 Mile)'}, "map_to": "1 Mile"},
        
        # Walks
        {"conditions": {"EVENT": r'1500m Race walk'}, "map_to": "1500m Racewalk"},
        {"conditions": {"EVENT": r'(3000m Race walk|3km Racewalk|3km Race Walk)'}, "map_to": "3000m Racewalk"},
        {"conditions": {"EVENT": r'(5000m Race Walk|5km Racewalk)'}, "map_to": "5000m Racewalk"},
        {"conditions": {"EVENT": r'(10km Race Walk|10km Racewalk|10,000m Racewalk)'}, "map_to": "10000m Racewalk"},
        {"conditions": {"EVENT": r'(20km Race Walk|20km Racewalk|20,000m Racewalk)'}, "map_to": "20km Racewalk"},
        {"conditions": {"EVENT": r'Race Walk', "DISTANCE": r'10000'}, "map_to": "10000m Racewalk"},
        
        # Relays
        {"conditions": {"EVENT": r'Relay', "DISTANCE": r'\b400\b'}, "map_to": "4 x 100m"},
        {"conditions": {"EVENT": r'Relay', "DISTANCE": r'\b1600\b'}, "map_to": "4 x 400m"},
        
        # Steeple
        {"conditions": {"EVENT": r'(3000m S/C|3000m SC|3000m Steeplechase)'}, "map_to": "3000m Steeplechase"},
        {"conditions": {"EVENT": r'(Steeplechase|S/C|SC)', "DISTANCE": r'3000'}, "map_to": "3000m Steeplechase"},
        {"conditions": {"EVENT": r'(2000m S/C|2000m SC|2000m Steeplechase)'}, "map_to": "2000m Steeplechase"},
        {"conditions": {"EVENT": r'(Steeplechase|S/C|SC)', "DISTANCE": r'2000'}, "map_to": "2000m Steeplechase"},

    ]

    distance_rules.append({
        "conditions": {"EVENT": r'Run|10,000|10000|10km|10 km', "DISTANCE": r'10,000|10000|10km|10 km'},
        "map_to": "10,000m"
    })

    distance_rules.append({
        "conditions": {"EVENT": r'Race Walk|Racewalk', "DISTANCE": r'10,000|10000|10km|10 km'},
        "map_to": "10,000m Racewalk"
    })

    distance_rules.append({
        "conditions": {"EVENT": r'Race Walk|Racewalk', "DISTANCE": r'5000|5km|5 km'},
        "map_to": "5000m Racewalk"
    })

    distance_rules.append({
        "conditions": {"EVENT": r'Race Walk|Racewalk', "DISTANCE": r'3000|3km|3 km'},
        "map_to": "3000m Racewalk"
    })


    
    for rule in distance_rules:
        cond = pd.Series(True, index=athletes.index)
        for col, pat in rule["conditions"].items():
            if col in athletes.columns:
                cond &= athletes[col].str.contains(pat, na=False, case=False, regex=True)
            else:
                cond &= False
        athletes.loc[cond, 'MAPPED_EVENT'] = rule["map_to"]

    # ----------------------
    # Hurdles
    # ----------------------
    hurdles_rules = [
        {"conditions": {"EVENT": r'(60m Hurdles|60m hurdles)'}, "map_to": "60m Hurdles"},
        {"conditions": {"EVENT": r'^Hurdles$', "DISTANCE": r'\b60\b'}, "map_to": "60m Hurdles"},
        {"conditions": {"EVENT": r'(100m Hurdles|100m hurdles)'}, "map_to": "100m Hurdles"},
        {"conditions": {"EVENT": r'^Hurdles$', "DISTANCE": r'\b100\b'}, "map_to": "100m Hurdles"},
        {"conditions": {"EVENT": r'(110m Hurdles|110m hurdles)'}, "map_to": "110m Hurdles"},
        {"conditions": {"EVENT": r'^Hurdles$', "DISTANCE": r'\b110\b'}, "map_to": "110m Hurdles"},
        {"conditions": {"EVENT": r'(200m Hurdles|200m hurdles)'}, "map_to": "200m Hurdles"},
        {"conditions": {"EVENT": r'^Hurdles$', "DISTANCE": r'\b200\b'}, "map_to": "200m Hurdles"},
        {"conditions": {"EVENT": r'(400m Hurdles|400m hurdles)'}, "map_to": "400m Hurdles"},
        {"conditions": {"EVENT": r'^Hurdles$', "DISTANCE": r'\b400\b'}, "map_to": "400m Hurdles"},
    ]

    for rule in hurdles_rules:
        cond = pd.Series(True, index=athletes.index)
        for col, pat in rule["conditions"].items():
            if col in athletes.columns:
                cond &= athletes[col].str.contains(pat, na=False, case=False, regex=True)
            else:
                cond &= False
        athletes.loc[cond, 'MAPPED_EVENT'] = rule["map_to"]

    # ----------------------
    # EVENT-only rules (regex on EVENT) - Field Events (THROWS & JUMPS)
    # ----------------------
    field_event_rules = {
        # Throws
        r'Discus\s*Throw|Discus$': 'Discus Throw',
        r'Shot\s*Put': 'Shot Put',
        r'Javelin\s*Throw|Javelin$': 'Javelin Throw',
        r'Hammer\s*Throw': 'Hammer Throw',

        # Jumps
        r'Long\s*Jump': 'Long Jump',
        r'Triple\s*Jump': 'Triple Jump',
        r'High\s*Jump': 'High Jump',
        r'Pole\s*Vault': 'Pole Vault',

        # Decathlon/Heptathlon
        r'Decathlon': 'Decathlon',
        r'Heptathlon': 'Heptathlon',
        
    }
    
    for pattern, mapped in field_event_rules.items():
        athletes.loc[athletes['EVENT'].str.contains(pattern, na=False, case=False), 'MAPPED_EVENT'] = mapped
   
    return athletes

athletes = simple_map_events(athletes)


/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_69858/4125795050.py:55: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  athletes.loc[athletes['EVENT'].str.contains(pattern, na=False, case=False), 'MAPPED_EVENT'] = mapped
/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_69858/4125795050.py:55: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '60m' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  athletes.loc[athletes['EVENT'].str.contains(pattern, na=False, case=False), 'MAPPED_EVENT'] = mapped
/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_69858/4125795050.py:55: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  athletes.loc[athletes['EVENT'].str.contains(pattern, n

In [116]:
athletes

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,YEAR,REGION,TIMESTAMP,NOW,delta_time,delta_time_conv,event_month,clean_name,DOB_parsed,MAPPED_EVENT
0,SI EN TABITHA NG,11:03.95,,,4.0,,3000m,,,,...,2025,International,2025-12-09 12:38:00+00:00,2025-12-22 16:43:34.679093+00:00,29 days 16:43:34.679093,29,11,si en tabitha ng,None,3000m
1,JE AN GARRETT CHUA,6.84,,,3.0,,Long Jump,,,,...,2025,International,2025-12-09 12:38:00+00:00,2025-12-22 16:43:34.679093+00:00,29 days 16:43:34.679093,29,11,je an garrett chua,None,Long Jump
2,LAUREL JIA EN LIM,27.71,,,6.0,,Discus Throw,,,,...,2025,International,2025-12-09 12:38:00+00:00,2025-12-22 16:43:34.679093+00:00,29 days 16:43:34.679093,29,11,laurel jia en lim,None,Discus Throw
3,JOSHUA SHYEN LEE,11.03,,,4.0,,100m,,,,...,2025,International,2025-12-09 12:38:00+00:00,2025-12-22 16:43:34.679093+00:00,29 days 16:43:34.679093,29,11,joshua shyen lee,None,100m
4,DARYEN XIN TZE KO,54.65,,,2.0,,400m Hurdles,,,,...,2025,International,2025-12-09 12:38:00+00:00,2025-12-22 16:43:34.679093+00:00,29 days 16:43:34.679093,29,11,daryen xin tze ko,None,400m Hurdles
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20254,"Ng, Zavier",9.88m,Hwa Chong Institution,14,18,U15,Triple Jump,0.0,None,Z854G11,...,2025,Local,2025-11-15 13:47:37.773700,2025-12-22 16:43:34.679093+00:00,295 days 16:43:34.679093,295,3,ng zavier,2011-03-14,Triple Jump
20255,"Choo Jia Yi, Allyson",9.30m,Team Start Singapore,17.0,3,Open,Triple Jump,0.0,None,A967D08,...,2025,Local,2025-11-15 13:49:23.923169,2025-12-22 16:43:34.679093+00:00,114 days 16:43:34.679093,114,8,choo jia yi allyson,2008-01-10,Triple Jump
20256,Lua Yu Xuan,10.02,NYGH,13.5,1.0,C,Triple Jump,,,,...,2025,Local,2025-11-17 17:34:00+00:00,2025-12-22 16:43:34.679093+00:00,248 days 16:43:34.679093,248,4,lua yu xuan,None,Triple Jump
20257,Muhammad Aaryan Shah Bin Azhar,12.61,SSP,15.5,3.0,B,Triple Jump,,,,...,2025,Local,2025-11-17 17:34:00+00:00,2025-12-22 16:43:34.679093+00:00,253 days 16:43:34.679093,253,4,muhammad aaryan shah bin azhar,None,Triple Jump


In [117]:
athletes[athletes['MAPPED_EVENT']=='']

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,YEAR,REGION,TIMESTAMP,NOW,delta_time,delta_time_conv,event_month,clean_name,DOB_parsed,MAPPED_EVENT


In [118]:
for col in athletes.columns:
    athletes[col] = athletes[col].astype(str)
    athletes[col] = athletes[col].str.replace('\xa0', ' ', regex=True)
    athletes[col] = athletes[col].str.replace('[\x00-\x1f\x7f-\x9f]', '', regex=True)
    athletes[col] = athletes[col].str.replace('\r', ' ', regex=True)
    athletes[col] = athletes[col].str.replace('\n', ' ', regex=True)
    athletes[col] = athletes[col].str.strip()

In [119]:
athletes[athletes['MAPPED_EVENT']!='']

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,YEAR,REGION,TIMESTAMP,NOW,delta_time,delta_time_conv,event_month,clean_name,DOB_parsed,MAPPED_EVENT
0,SI EN TABITHA NG,11:03.95,,,4.0,,3000m,,,,...,2025,International,2025-12-09 12:38:00+00:00,2025-12-22 16:43:34.679093+00:00,29 days 16:43:34.679093,29,11,si en tabitha ng,None,3000m
1,JE AN GARRETT CHUA,6.84,,,3.0,,Long Jump,,,,...,2025,International,2025-12-09 12:38:00+00:00,2025-12-22 16:43:34.679093+00:00,29 days 16:43:34.679093,29,11,je an garrett chua,None,Long Jump
2,LAUREL JIA EN LIM,27.71,,,6.0,,Discus Throw,,,,...,2025,International,2025-12-09 12:38:00+00:00,2025-12-22 16:43:34.679093+00:00,29 days 16:43:34.679093,29,11,laurel jia en lim,None,Discus Throw
3,JOSHUA SHYEN LEE,11.03,,,4.0,,100m,,,,...,2025,International,2025-12-09 12:38:00+00:00,2025-12-22 16:43:34.679093+00:00,29 days 16:43:34.679093,29,11,joshua shyen lee,None,100m
4,DARYEN XIN TZE KO,54.65,,,2.0,,400m Hurdles,,,,...,2025,International,2025-12-09 12:38:00+00:00,2025-12-22 16:43:34.679093+00:00,29 days 16:43:34.679093,29,11,daryen xin tze ko,None,400m Hurdles
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20254,"Ng, Zavier",9.88m,Hwa Chong Institution,14,18,U15,Triple Jump,0.0,None,Z854G11,...,2025,Local,2025-11-15 13:47:37.773700,2025-12-22 16:43:34.679093+00:00,295 days 16:43:34.679093,295,3,ng zavier,2011-03-14,Triple Jump
20255,"Choo Jia Yi, Allyson",9.30m,Team Start Singapore,17.0,3,Open,Triple Jump,0.0,None,A967D08,...,2025,Local,2025-11-15 13:49:23.923169,2025-12-22 16:43:34.679093+00:00,114 days 16:43:34.679093,114,8,choo jia yi allyson,2008-01-10,Triple Jump
20256,Lua Yu Xuan,10.02,NYGH,13.5,1.0,C,Triple Jump,,,,...,2025,Local,2025-11-17 17:34:00+00:00,2025-12-22 16:43:34.679093+00:00,248 days 16:43:34.679093,248,4,lua yu xuan,None,Triple Jump
20257,Muhammad Aaryan Shah Bin Azhar,12.61,SSP,15.5,3.0,B,Triple Jump,,,,...,2025,Local,2025-11-17 17:34:00+00:00,2025-12-22 16:43:34.679093+00:00,253 days 16:43:34.679093,253,4,muhammad aaryan shah bin azhar,None,Triple Jump


In [120]:
athletes

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,YEAR,REGION,TIMESTAMP,NOW,delta_time,delta_time_conv,event_month,clean_name,DOB_parsed,MAPPED_EVENT
0,SI EN TABITHA NG,11:03.95,,,4.0,,3000m,,,,...,2025,International,2025-12-09 12:38:00+00:00,2025-12-22 16:43:34.679093+00:00,29 days 16:43:34.679093,29,11,si en tabitha ng,None,3000m
1,JE AN GARRETT CHUA,6.84,,,3.0,,Long Jump,,,,...,2025,International,2025-12-09 12:38:00+00:00,2025-12-22 16:43:34.679093+00:00,29 days 16:43:34.679093,29,11,je an garrett chua,None,Long Jump
2,LAUREL JIA EN LIM,27.71,,,6.0,,Discus Throw,,,,...,2025,International,2025-12-09 12:38:00+00:00,2025-12-22 16:43:34.679093+00:00,29 days 16:43:34.679093,29,11,laurel jia en lim,None,Discus Throw
3,JOSHUA SHYEN LEE,11.03,,,4.0,,100m,,,,...,2025,International,2025-12-09 12:38:00+00:00,2025-12-22 16:43:34.679093+00:00,29 days 16:43:34.679093,29,11,joshua shyen lee,None,100m
4,DARYEN XIN TZE KO,54.65,,,2.0,,400m Hurdles,,,,...,2025,International,2025-12-09 12:38:00+00:00,2025-12-22 16:43:34.679093+00:00,29 days 16:43:34.679093,29,11,daryen xin tze ko,None,400m Hurdles
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20254,"Ng, Zavier",9.88m,Hwa Chong Institution,14,18,U15,Triple Jump,0.0,None,Z854G11,...,2025,Local,2025-11-15 13:47:37.773700,2025-12-22 16:43:34.679093+00:00,295 days 16:43:34.679093,295,3,ng zavier,2011-03-14,Triple Jump
20255,"Choo Jia Yi, Allyson",9.30m,Team Start Singapore,17.0,3,Open,Triple Jump,0.0,None,A967D08,...,2025,Local,2025-11-15 13:49:23.923169,2025-12-22 16:43:34.679093+00:00,114 days 16:43:34.679093,114,8,choo jia yi allyson,2008-01-10,Triple Jump
20256,Lua Yu Xuan,10.02,NYGH,13.5,1.0,C,Triple Jump,,,,...,2025,Local,2025-11-17 17:34:00+00:00,2025-12-22 16:43:34.679093+00:00,248 days 16:43:34.679093,248,4,lua yu xuan,None,Triple Jump
20257,Muhammad Aaryan Shah Bin Azhar,12.61,SSP,15.5,3.0,B,Triple Jump,,,,...,2025,Local,2025-11-17 17:34:00+00:00,2025-12-22 16:43:34.679093+00:00,253 days 16:43:34.679093,253,4,muhammad aaryan shah bin azhar,None,Triple Jump


In [121]:
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/SEAG_u18/')

athletes.to_csv('check.csv', encoding='utf-8')

## Load Benchmarks

In [122]:
# Load Benchmarks

os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/Benchmarks/')

benchmarks = pd.read_csv("benchmarks_SEAG2025.csv")


In [123]:
benchmarks

,EVENT,GENDER,BENCHMARK
0,100m,Male,10.26
1,200m,Male,20.73
2,400m,Male,46.21
3,800m,Male,01:49.9
4,1500m,Male,03:50.1
5,5000m,Male,14:48.0
6,"10,000m",Male,29:54.6
7,110m Hurdles,Male,13.85
8,400m Hurdles,Male,50.91
9,3000m Steeplechase,Male,08:58.3


In [124]:
benchmarks

,EVENT,GENDER,BENCHMARK
0,100m,Male,10.26
1,200m,Male,20.73
2,400m,Male,46.21
3,800m,Male,01:49.9
4,1500m,Male,03:50.1
5,5000m,Male,14:48.0
6,"10,000m",Male,29:54.6
7,110m Hurdles,Male,13.85
8,400m Hurdles,Male,50.91
9,3000m Steeplechase,Male,08:58.3


In [125]:

# Handles 00:XX.XX format and converts into XX.XX

def convert_time_refactored(i, string, metric):
    """
    Convert various metric formats (distance, time) to a float value (primarily seconds for times).
    Optimized for speed: no global variables, no print statements, no unnecessary conversions.
    
    Args:
        i (int): Index (unused, kept for compatibility).
        string (str): Event description.
        metric (str, float, or datetime): The result metric.
    
    Returns:
        float or empty string: Converted metric as float (seconds/meters), or '' if not convertible.
    """
    l = ['discus', 'throw', 'jump', 'vault', 'shot']
    sprint_events = ['100m', '200m', '400m']
    
    string = str(string).lower()
    metric_str = str(metric)
    output = ''
    
    try:
        # Skip marks with illegal wind speeds
        if isinstance(metric_str, str) and 'w' in metric_str.lower():
            return ''
        
        # Field events (distances)
        if any(s in string for s in l):
            # Remove unit if present
            metric_clean = metric_str.replace('m', '').replace('GR', '')
            return round(float(metric_clean), 2)
        
        # No event description
        if string == '':
            return ''
        
        # Time events
        count_colon = metric_str.count(':')
        count_dot = metric_str.count('.')
        
        # Simple time as float (no colon)
        if count_colon == 0:
            return round(float(metric_str), 2)
        
        # Sprint events (100m, 200m, 400m): Handle 00:MM.SS or MM.SS format as seconds
        if any(sprint in string for sprint in sprint_events):
            if count_colon == 1 and count_dot == 1:
                # Format: 00:09.16 or 09.16
                parts = metric_str.split(':')
                if len(parts) == 2:
                    # Check if first part is "00" (ignore it) or actual minutes
                    first_part = parts[0]
                    second_part = parts[1]
                    
                    if first_part == '00':
                        # It's 00:SS.ss format, just return the seconds part
                        return float(second_part)
                    else:
                        # It's MM:SS.ss format, convert normally
                        return float(int(first_part) * 60 + float(second_part))
        
        # Convert time formats with two colons (like XX:XX:XX, XX:XX.XX)
        if count_colon == 2:

            # Check if this is a standard HH:MM:SS (no decimal point)
            if count_dot == 0:
                h, m, s = metric_str.split(':')
                return float(
                    int(h) * 3600 + int(m) * 60 + float(s)
                )
            # For 10,000m, 5000m, and 1500m, replace the 6th character with '.' for format XX:XX.XX
            if ('10,000m' in string or '5000m' in string or '1500m' in string):
                if len(metric_str) == 7:  # X:XX:XX (1500m special case)
                    idx = 4
                    metric_mod = '0' + metric_str[:idx] + '.' + metric_str[idx+1:]
                else:
                    idx = 5
                    metric_mod = metric_str[:idx] + '.' + metric_str[idx+1:]
                m, s = metric_mod.split(':')[-2:]
                return float((int(m) * 60) + float(s))
            
            # Standard HH:MM:SS
            h, m, s = metric_str.split(':')
            return float(int(h) * 3600 + int(m) * 60 + float(s))
        
        # Handle datetime.time/datetime.datetime objects
        if isinstance(metric, (datetime.time, datetime.datetime)):
            t = str(metric)
            h, m, s = t.split(':')
            return float(int(h) * 3600 + int(m) * 60 + float(s))
        
        # MM:SS.sss format
        if count_colon == 1 and count_dot >= 1:
            m, s = metric_str.split(':')
            return float(int(m) * 60 + float(s))
        
        # HH.MM.SS (rare) or MM:SS:SS
        if count_colon == 1 and count_dot == 2:
            # Replace first dot with colon
            metric_mod = metric_str.replace('.', ':', 1)
            h, m, s = metric_mod.split(':')
            return float(int(h) * 3600 + int(m) * 60 + float(s))
        
        # HH:MM:SS (no dots)
        if count_colon == 2 and count_dot == 0:
            h, m, s = metric_str.split(':')
            return float(int(h) * 3600 + int(m) * 60 + float(s))
        
        # MM:SS (no dots)
        if count_colon == 1 and count_dot == 0:
            m, s = metric_str.split(':')
            return float(int(m) * 60 + int(s))
            
    except Exception:
        return ''
    
    return output



In [126]:
def process_benchmarks(df):
    
    for i in range(len(df)):

        rowIndex = df.index[i]

        input_string=df.iloc[rowIndex,0]
    
        metric=df.iloc[rowIndex,2]
    
        if metric==None:
        
            continue
        
        out = convert_time_refactored(i, input_string, metric)
        
        print(rowIndex, input_string, out)

    
        df.loc[rowIndex, 'Metric'] = out
    
    return df

In [127]:
process_benchmarks(benchmarks)

0 100m 10.26
1 200m 20.73
2 400m 46.21
3 800m 109.9
4 1500m 230.1
5 5000m 888.0
6 10,000m 1794.6
7 110m Hurdles 13.85
8 400m Hurdles 50.91
9 3000m Steeplechase 538.3
10 4 x 100m 39.51
11 4 x 400m 190.7
12 Marathon 9089.0
13 20km Racewalk 5877.0
14 High Jump 2.19
15 Pole Vault 5.2
16 Long Jump 7.53
17 Triple Jump 16.09
18 Shot Put 16.66
19 Discus Throw 53.34
20 Hammer Throw 59.81
21 Javelin Throw 69.62
22 Decathlon 6582.0
23 100m 11.58
24 200m 23.5
25 400m 53.4
26 800m 130.6
27 1500m 278.7
28 5000m 1029.9
29 10,000m 2133.2
30 100m Hurdles 13.43
31 400m Hurdles 57.75
32 3000m Steeplechase 655.7
33 4 x 100m 43.97
34 4 x 400m 218.9
35 Marathon 3280.0
36 20km Racewalk 6495.0
37 High Jump 1.75
38 Pole Vault 3.9
39 Long Jump 6.27
40 Triple Jump 13.68
41 Shot Put 15.92
42 Discus Throw 49.34
43 Hammer Throw 56.27
44 Javelin Throw 51.66
45 Heptathlon 5201.0


,EVENT,GENDER,BENCHMARK,Metric
0,100m,Male,10.26,10.26
1,200m,Male,20.73,20.73
2,400m,Male,46.21,46.21
3,800m,Male,01:49.9,109.90
4,1500m,Male,03:50.1,230.10
5,5000m,Male,14:48.0,888.00
6,"10,000m",Male,29:54.6,1794.60
7,110m Hurdles,Male,13.85,13.85
8,400m Hurdles,Male,50.91,50.91
9,3000m Steeplechase,Male,08:58.3,538.30


In [128]:
benchmarks

,EVENT,GENDER,BENCHMARK,Metric
0,100m,Male,10.26,10.26
1,200m,Male,20.73,20.73
2,400m,Male,46.21,46.21
3,800m,Male,01:49.9,109.90
4,1500m,Male,03:50.1,230.10
5,5000m,Male,14:48.0,888.00
6,"10,000m",Male,29:54.6,1794.60
7,110m Hurdles,Male,13.85,13.85
8,400m Hurdles,Male,50.91,50.91
9,3000m Steeplechase,Male,08:58.3,538.30


In [129]:
mask = benchmarks['EVENT'].str.lower().str.contains(r'jump|throw|put|pole|decathlon|heptathlon', na=True)

benchmarks.loc[mask, '2%']=benchmarks['Metric']*0.98
benchmarks.loc[mask, '3.5%']=benchmarks['Metric']*0.965
benchmarks.loc[mask, '5%']=benchmarks['Metric']*0.95
benchmarks.loc[mask, '10%']=benchmarks['Metric']*0.90


benchmarks.loc[~mask, '2%']=benchmarks['Metric']*1.02
benchmarks.loc[~mask, '3.5%']=benchmarks['Metric']*1.035
benchmarks.loc[~mask, '5%']=benchmarks['Metric']*1.05
benchmarks.loc[~mask, '10%']=benchmarks['Metric']*1.10


In [130]:
benchmarks['MAPPED_EVENT']=benchmarks['EVENT'].str.strip()

In [131]:
for col in benchmarks.columns:
    benchmarks[col] = benchmarks[col].astype(str)
    benchmarks[col] = benchmarks[col].str.replace('\xa0', ' ', regex=True)
    benchmarks[col] = benchmarks[col].str.replace('[\x00-\x1f\x7f-\x9f]', '', regex=True)
    benchmarks[col] = benchmarks[col].str.replace('\r', ' ', regex=True)
    benchmarks[col] = benchmarks[col].str.replace('\n', ' ', regex=True)
    benchmarks[col] = benchmarks[col].str.strip()


In [132]:
benchmarks.head(50)

,EVENT,GENDER,BENCHMARK,Metric,2%,3.5%,5%,10%,MAPPED_EVENT
0,100m,Male,10.26,10.26,10.4652,10.6191,10.773,11.286000000000001,100m
1,200m,Male,20.73,20.73,21.1446,21.45555,21.7665,22.803,200m
2,400m,Male,46.21,46.21,47.1342,47.827349999999996,48.520500000000006,50.831,400m
3,800m,Male,01:49.9,109.9,112.09800000000001,113.7465,115.39500000000001,120.89000000000001,800m
4,1500m,Male,03:50.1,230.1,234.702,238.15349999999998,241.60500000000002,253.11,1500m
5,5000m,Male,14:48.0,888.0,905.76,919.0799999999999,932.4000000000001,976.8000000000001,5000m
6,"10,000m",Male,29:54.6,1794.6,1830.492,1857.4109999999998,1884.33,1974.0600000000002,"10,000m"
7,110m Hurdles,Male,13.85,13.85,14.127,14.334749999999998,14.5425,15.235000000000001,110m Hurdles
8,400m Hurdles,Male,50.91,50.91,51.9282,52.691849999999995,53.4555,56.001,400m Hurdles
9,3000m Steeplechase,Male,08:58.3,538.3,549.0659999999999,557.1404999999999,565.215,592.13,3000m Steeplechase


In [133]:
athletes

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,YEAR,REGION,TIMESTAMP,NOW,delta_time,delta_time_conv,event_month,clean_name,DOB_parsed,MAPPED_EVENT
0,SI EN TABITHA NG,11:03.95,,,4.0,,3000m,,,,...,2025,International,2025-12-09 12:38:00+00:00,2025-12-22 16:43:34.679093+00:00,29 days 16:43:34.679093,29,11,si en tabitha ng,None,3000m
1,JE AN GARRETT CHUA,6.84,,,3.0,,Long Jump,,,,...,2025,International,2025-12-09 12:38:00+00:00,2025-12-22 16:43:34.679093+00:00,29 days 16:43:34.679093,29,11,je an garrett chua,None,Long Jump
2,LAUREL JIA EN LIM,27.71,,,6.0,,Discus Throw,,,,...,2025,International,2025-12-09 12:38:00+00:00,2025-12-22 16:43:34.679093+00:00,29 days 16:43:34.679093,29,11,laurel jia en lim,None,Discus Throw
3,JOSHUA SHYEN LEE,11.03,,,4.0,,100m,,,,...,2025,International,2025-12-09 12:38:00+00:00,2025-12-22 16:43:34.679093+00:00,29 days 16:43:34.679093,29,11,joshua shyen lee,None,100m
4,DARYEN XIN TZE KO,54.65,,,2.0,,400m Hurdles,,,,...,2025,International,2025-12-09 12:38:00+00:00,2025-12-22 16:43:34.679093+00:00,29 days 16:43:34.679093,29,11,daryen xin tze ko,None,400m Hurdles
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20254,"Ng, Zavier",9.88m,Hwa Chong Institution,14,18,U15,Triple Jump,0.0,None,Z854G11,...,2025,Local,2025-11-15 13:47:37.773700,2025-12-22 16:43:34.679093+00:00,295 days 16:43:34.679093,295,3,ng zavier,2011-03-14,Triple Jump
20255,"Choo Jia Yi, Allyson",9.30m,Team Start Singapore,17.0,3,Open,Triple Jump,0.0,None,A967D08,...,2025,Local,2025-11-15 13:49:23.923169,2025-12-22 16:43:34.679093+00:00,114 days 16:43:34.679093,114,8,choo jia yi allyson,2008-01-10,Triple Jump
20256,Lua Yu Xuan,10.02,NYGH,13.5,1.0,C,Triple Jump,,,,...,2025,Local,2025-11-17 17:34:00+00:00,2025-12-22 16:43:34.679093+00:00,248 days 16:43:34.679093,248,4,lua yu xuan,None,Triple Jump
20257,Muhammad Aaryan Shah Bin Azhar,12.61,SSP,15.5,3.0,B,Triple Jump,,,,...,2025,Local,2025-11-17 17:34:00+00:00,2025-12-22 16:43:34.679093+00:00,253 days 16:43:34.679093,253,4,muhammad aaryan shah bin azhar,None,Triple Jump


## Merge Bencharks with Athletes

In [134]:
# Merge benchmarks onto athletes on MAPPED_EVENT and GENDER

df = pd.merge(
    left=athletes, 
    right=benchmarks,
    how='left',
    left_on=['MAPPED_EVENT', 'GENDER'],
    right_on=['MAPPED_EVENT', 'GENDER'],
)

In [135]:
df

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT_x,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,clean_name,DOB_parsed,MAPPED_EVENT,EVENT_y,BENCHMARK,Metric,2%,3.5%,5%,10%
0,SI EN TABITHA NG,11:03.95,,,4.0,,3000m,,,,...,si en tabitha ng,None,3000m,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,JE AN GARRETT CHUA,6.84,,,3.0,,Long Jump,,,,...,je an garrett chua,None,Long Jump,Long Jump,7.53 m,7.53,7.3794,7.26645,7.1535,6.777
2,LAUREL JIA EN LIM,27.71,,,6.0,,Discus Throw,,,,...,laurel jia en lim,None,Discus Throw,Discus Throw,49.34 m,49.34,48.3532,47.6131,46.873,44.406000000000006
3,JOSHUA SHYEN LEE,11.03,,,4.0,,100m,,,,...,joshua shyen lee,None,100m,100m,10.26,10.26,10.4652,10.6191,10.773,11.286000000000001
4,DARYEN XIN TZE KO,54.65,,,2.0,,400m Hurdles,,,,...,daryen xin tze ko,None,400m Hurdles,400m Hurdles,50.91,50.91,51.9282,52.691849999999995,53.4555,56.001
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20254,"Ng, Zavier",9.88m,Hwa Chong Institution,14,18,U15,Triple Jump,0.0,None,Z854G11,...,ng zavier,2011-03-14,Triple Jump,Triple Jump,16.09 m,16.09,15.7682,15.52685,15.285499999999999,14.481
20255,"Choo Jia Yi, Allyson",9.30m,Team Start Singapore,17.0,3,Open,Triple Jump,0.0,None,A967D08,...,choo jia yi allyson,2008-01-10,Triple Jump,Triple Jump,13.68 m,13.68,13.4064,13.2012,12.995999999999999,12.312
20256,Lua Yu Xuan,10.02,NYGH,13.5,1.0,C,Triple Jump,,,,...,lua yu xuan,None,Triple Jump,Triple Jump,13.68 m,13.68,13.4064,13.2012,12.995999999999999,12.312
20257,Muhammad Aaryan Shah Bin Azhar,12.61,SSP,15.5,3.0,B,Triple Jump,,,,...,muhammad aaryan shah bin azhar,None,Triple Jump,Triple Jump,16.09 m,16.09,15.7682,15.52685,15.285499999999999,14.481


In [136]:
# Identify rows causing left join to produce larger dataframe

duplicates_in_right = benchmarks[benchmarks.duplicated(subset=['MAPPED_EVENT', 'GENDER'], keep=False)]


In [137]:
duplicates_in_right

,EVENT,GENDER,BENCHMARK,Metric,2%,3.5%,5%,10%,MAPPED_EVENT


In [138]:
#left_df_with_duplicates = athletes[athletes['MAPPED_EVENT', 'GENDER'].isin(duplicates_in_right['MAPPED_EVENT', 'GENDER'])]

In [139]:
df[df['MAPPED_EVENT']=='Decathlon']

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT_x,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,clean_name,DOB_parsed,MAPPED_EVENT,EVENT_y,BENCHMARK,Metric,2%,3.5%,5%,10%
2987,Lucas Le Cong Fun,1.88,,,1.0,,Decathlon,,,,...,lucas le cong fun,None,Decathlon,Decathlon,6582,6582.0,6450.36,6351.63,6252.9,5923.8
4067,Lucas Le Cong Fun,35.65m,,,1.0,,Decathlon,,,,...,lucas le cong fun,None,Decathlon,Decathlon,6582,6582.0,6450.36,6351.63,6252.9,5923.8
4430,Lucas Le Cong Fun,16.01,,,2.0,,Decathlon,,,,...,lucas le cong fun,None,Decathlon,Decathlon,6582,6582.0,6450.36,6351.63,6252.9,5923.8
4431,Lucas Le Cong Fun,5958,,,,,Decathlon,,,,...,lucas le cong fun,2006-01-11,Decathlon,Decathlon,6582,6582.0,6450.36,6351.63,6252.9,5923.8
11427,Lucas Le Cong Fun,5958,,,,,Decathlon,,,,...,lucas le cong fun,2006-01-11,Decathlon,Decathlon,6582,6582.0,6450.36,6351.63,6252.9,5923.8
13416,Lucas Le Cong Fun,11.00m,,,1.0,,Decathlon,,,,...,lucas le cong fun,None,Decathlon,Decathlon,6582,6582.0,6450.36,6351.63,6252.9,5923.8
14128,Lucas Le Cong Fun,55.57,,,5.0,,Decathlon,,,,...,lucas le cong fun,None,Decathlon,Decathlon,6582,6582.0,6450.36,6351.63,6252.9,5923.8
15177,Lucas Le Cong Fun,5.87,,,4.0,,Decathlon,,,,...,lucas le cong fun,None,Decathlon,Decathlon,6582,6582.0,6450.36,6351.63,6252.9,5923.8
15848,Jayden Ng,5716,,,3,U18,Decathlon,,,,...,jayden ng,2008-12-22,Decathlon,Decathlon,6582,6582.0,6450.36,6351.63,6252.9,5923.8
16522,Lucas Le Cong Fun,49.8,,,1.0,,Decathlon,,,,...,lucas le cong fun,None,Decathlon,Decathlon,6582,6582.0,6450.36,6351.63,6252.9,5923.8


In [140]:
# replace '-' with NaN

df['RESULT'] = df['RESULT'].replace(regex=r'–', value=np.NaN)
#df['SEED'] = df['SEED'].replace(regex=r'–', value=np.NaN)


In [141]:
df

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT_x,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,clean_name,DOB_parsed,MAPPED_EVENT,EVENT_y,BENCHMARK,Metric,2%,3.5%,5%,10%
0,SI EN TABITHA NG,11:03.95,,,4.0,,3000m,,,,...,si en tabitha ng,None,3000m,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,JE AN GARRETT CHUA,6.84,,,3.0,,Long Jump,,,,...,je an garrett chua,None,Long Jump,Long Jump,7.53 m,7.53,7.3794,7.26645,7.1535,6.777
2,LAUREL JIA EN LIM,27.71,,,6.0,,Discus Throw,,,,...,laurel jia en lim,None,Discus Throw,Discus Throw,49.34 m,49.34,48.3532,47.6131,46.873,44.406000000000006
3,JOSHUA SHYEN LEE,11.03,,,4.0,,100m,,,,...,joshua shyen lee,None,100m,100m,10.26,10.26,10.4652,10.6191,10.773,11.286000000000001
4,DARYEN XIN TZE KO,54.65,,,2.0,,400m Hurdles,,,,...,daryen xin tze ko,None,400m Hurdles,400m Hurdles,50.91,50.91,51.9282,52.691849999999995,53.4555,56.001
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20254,"Ng, Zavier",9.88m,Hwa Chong Institution,14,18,U15,Triple Jump,0.0,None,Z854G11,...,ng zavier,2011-03-14,Triple Jump,Triple Jump,16.09 m,16.09,15.7682,15.52685,15.285499999999999,14.481
20255,"Choo Jia Yi, Allyson",9.30m,Team Start Singapore,17.0,3,Open,Triple Jump,0.0,None,A967D08,...,choo jia yi allyson,2008-01-10,Triple Jump,Triple Jump,13.68 m,13.68,13.4064,13.2012,12.995999999999999,12.312
20256,Lua Yu Xuan,10.02,NYGH,13.5,1.0,C,Triple Jump,,,,...,lua yu xuan,None,Triple Jump,Triple Jump,13.68 m,13.68,13.4064,13.2012,12.995999999999999,12.312
20257,Muhammad Aaryan Shah Bin Azhar,12.61,SSP,15.5,3.0,B,Triple Jump,,,,...,muhammad aaryan shah bin azhar,None,Triple Jump,Triple Jump,16.09 m,16.09,15.7682,15.52685,15.285499999999999,14.481


In [142]:
df[df['NAME']=='Caleb Hia']

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT_x,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,clean_name,DOB_parsed,MAPPED_EVENT,EVENT_y,BENCHMARK,Metric,2%,3.5%,5%,10%
85,Caleb Hia,2:29:21,<NA>,<NA>,17.0,<NA>,Marathon,<NA>,<NA>,<NA>,...,caleb hia,1992-12-22,Marathon,Marathon,02:31:29,9089.0,9270.78,9407.115,9543.45,9997.900000000001


In [143]:
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/SEAG_u18/')


df.to_csv('seag_u18_postmap.csv', sep=',', encoding='utf-8-sig', index=False)


## Normalize Results and Determine Performance Tolerances

In [144]:
# Convert results and seed into seconds format for mapped events only (vectorised version)

# First, normalize data as you did (remove control chars etc.)
for col in df.columns:
    df[col] = df[col].astype(str)
    df[col] = df[col].str.replace('\xa0', ' ', regex=True)
    df[col] = df[col].str.replace('[\x00-\x1f\x7f-\x9f]', '', regex=True)
    df[col] = df[col].str.replace('\r', ' ', regex=True)
    df[col] = df[col].str.replace('\n', ' ', regex=True)
    df[col] = df[col].str.strip()

# Define a filter for rows with convertible results
invalid_results = {'—', 'None', 'DQ', 'SCR', 'FS', 'DNQ', 'DNS', 'NH', 'NM', 'FOUL', 'DNF', 'SR'}

# Apply conversion vectorized using apply, skipping invalid values
def convert_for_row(row):
    if row['RESULT'] in invalid_results:
        return ''
    return convert_time_refactored(row.name, row['MAPPED_EVENT'], row['RESULT'])

df['RESULT_CONV'] = df.apply(convert_for_row, axis=1)


In [145]:
df

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT_x,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,DOB_parsed,MAPPED_EVENT,EVENT_y,BENCHMARK,Metric,2%,3.5%,5%,10%,RESULT_CONV
0,SI EN TABITHA NG,11:03.95,,,4.0,,3000m,,,,...,None,3000m,nan,nan,nan,nan,nan,nan,nan,663.95
1,JE AN GARRETT CHUA,6.84,,,3.0,,Long Jump,,,,...,None,Long Jump,Long Jump,7.53 m,7.53,7.3794,7.26645,7.1535,6.777,6.84
2,LAUREL JIA EN LIM,27.71,,,6.0,,Discus Throw,,,,...,None,Discus Throw,Discus Throw,49.34 m,49.34,48.3532,47.6131,46.873,44.406000000000006,27.71
3,JOSHUA SHYEN LEE,11.03,,,4.0,,100m,,,,...,None,100m,100m,10.26,10.26,10.4652,10.6191,10.773,11.286000000000001,11.03
4,DARYEN XIN TZE KO,54.65,,,2.0,,400m Hurdles,,,,...,None,400m Hurdles,400m Hurdles,50.91,50.91,51.9282,52.691849999999995,53.4555,56.001,54.65
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20254,"Ng, Zavier",9.88m,Hwa Chong Institution,14,18,U15,Triple Jump,0.0,None,Z854G11,...,2011-03-14,Triple Jump,Triple Jump,16.09 m,16.09,15.7682,15.52685,15.285499999999999,14.481,9.88
20255,"Choo Jia Yi, Allyson",9.30m,Team Start Singapore,17.0,3,Open,Triple Jump,0.0,None,A967D08,...,2008-01-10,Triple Jump,Triple Jump,13.68 m,13.68,13.4064,13.2012,12.995999999999999,12.312,9.3
20256,Lua Yu Xuan,10.02,NYGH,13.5,1.0,C,Triple Jump,,,,...,None,Triple Jump,Triple Jump,13.68 m,13.68,13.4064,13.2012,12.995999999999999,12.312,10.02
20257,Muhammad Aaryan Shah Bin Azhar,12.61,SSP,15.5,3.0,B,Triple Jump,,,,...,None,Triple Jump,Triple Jump,16.09 m,16.09,15.7682,15.52685,15.285499999999999,14.481,12.61


In [146]:
# Choose SEED if better than RESULT

#condition1=df['SEED_CONV']>df['RESULT_CONV']
#condition2=((df['CATEGORY_EVENT']=='Jump')|(df['CATEGORY_EVENT']=='Throw'))
#condition3=df['SEED_CONV']<df['RESULT_CONV']
#condition4=~((df['CATEGORY_EVENT']=='Jump')|(df['CATEGORY_EVENT']=='Throw'))


#df['RESULT_BEST']=df['SEED_CONV'].where((condition1 & condition2)|(condition3 & condition4), df['RESULT_CONV'].values)

df['RESULT_BEST'] = df['RESULT_CONV']

In [147]:
df

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT_x,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,MAPPED_EVENT,EVENT_y,BENCHMARK,Metric,2%,3.5%,5%,10%,RESULT_CONV,RESULT_BEST
0,SI EN TABITHA NG,11:03.95,,,4.0,,3000m,,,,...,3000m,nan,nan,nan,nan,nan,nan,nan,663.95,663.95
1,JE AN GARRETT CHUA,6.84,,,3.0,,Long Jump,,,,...,Long Jump,Long Jump,7.53 m,7.53,7.3794,7.26645,7.1535,6.777,6.84,6.84
2,LAUREL JIA EN LIM,27.71,,,6.0,,Discus Throw,,,,...,Discus Throw,Discus Throw,49.34 m,49.34,48.3532,47.6131,46.873,44.406000000000006,27.71,27.71
3,JOSHUA SHYEN LEE,11.03,,,4.0,,100m,,,,...,100m,100m,10.26,10.26,10.4652,10.6191,10.773,11.286000000000001,11.03,11.03
4,DARYEN XIN TZE KO,54.65,,,2.0,,400m Hurdles,,,,...,400m Hurdles,400m Hurdles,50.91,50.91,51.9282,52.691849999999995,53.4555,56.001,54.65,54.65
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20254,"Ng, Zavier",9.88m,Hwa Chong Institution,14,18,U15,Triple Jump,0.0,None,Z854G11,...,Triple Jump,Triple Jump,16.09 m,16.09,15.7682,15.52685,15.285499999999999,14.481,9.88,9.88
20255,"Choo Jia Yi, Allyson",9.30m,Team Start Singapore,17.0,3,Open,Triple Jump,0.0,None,A967D08,...,Triple Jump,Triple Jump,13.68 m,13.68,13.4064,13.2012,12.995999999999999,12.312,9.3,9.3
20256,Lua Yu Xuan,10.02,NYGH,13.5,1.0,C,Triple Jump,,,,...,Triple Jump,Triple Jump,13.68 m,13.68,13.4064,13.2012,12.995999999999999,12.312,10.02,10.02
20257,Muhammad Aaryan Shah Bin Azhar,12.61,SSP,15.5,3.0,B,Triple Jump,,,,...,Triple Jump,Triple Jump,16.09 m,16.09,15.7682,15.52685,15.285499999999999,14.481,12.61,12.61


In [148]:
# Change to numeric

df[['2%', '3.5%', '5%', '10%', 'RESULT_BEST', 'Metric']] = df[['2%', '3.5%', '5%', '10%', 'RESULT_BEST', 'Metric']].apply(pd.to_numeric, errors='coerce')

In [149]:
mask = df['CATEGORY_EVENT'].str.lower().str.contains(r'jump|throw|decathlon|heptathlon', na=True)

df.loc[mask, 'Delta2'] = df['RESULT_BEST']-df['2%']
df.loc[mask, 'Delta3.5'] = df['RESULT_BEST']-df['3.5%']
df.loc[mask, 'Delta5'] = df['RESULT_BEST']-df['5%']
df.loc[mask, 'Delta10'] = df['RESULT_BEST']-df['10%']
df.loc[mask, 'Delta_Benchmark'] = df['RESULT_BEST']-df['Metric']

df.loc[~mask, 'Delta2'] =  df['2%'] - df['RESULT_BEST']
df.loc[~mask, 'Delta3.5'] = df['3.5%'] - df['RESULT_BEST']
df.loc[~mask, 'Delta5'] = df['5%'] - df['RESULT_BEST']
df.loc[~mask, 'Delta10'] = df['10%'] - df['RESULT_BEST']
df.loc[~mask, 'Delta_Benchmark'] = df['Metric'] - df['RESULT_BEST']



In [150]:
# Performance metric to filter out athletes

df['PERF_SCALAR']=df['Delta5']/df['Metric']*100

In [151]:
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/SEAG_u18/')


df.to_csv('seag_u18_postmap_benchmarked.csv', sep=',', encoding='utf-8-sig', index=False)


In [152]:
df[df['MAPPED_EVENT']=='10,000m']

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT_x,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,5%,10%,RESULT_CONV,RESULT_BEST,Delta2,Delta3.5,Delta5,Delta10,Delta_Benchmark,PERF_SCALAR
34,Rui Yong Soh,31:31.91,<NA>,<NA>,7.0,<NA>,"10,000m",<NA>,<NA>,<NA>,...,1884.33,1974.06,1891.91,1891.91,-61.418,-34.499,-7.58,82.15,-97.31,-0.422378
35,Shaun Goh,31:45.26,<NA>,<NA>,8.0,<NA>,"10,000m",<NA>,<NA>,<NA>,...,1884.33,1974.06,1905.26,1905.26,-74.768,-47.849,-20.93,68.80,-110.66,-1.166277
3097,"Tan, Bernice",44:25.61,Lacticbuds,25.0,3,Open,Run,10000.0,None,B075C00,...,2239.86,2346.52,2665.61,2665.61,-489.746,-457.748,-425.75,-319.09,-532.41,-19.958279
3102,"Cheok, Samuel",39:21.56,TeamFabian,22,10,Open,Run,10000.0,None,S521D03,...,1884.33,1974.06,2361.56,2361.56,-531.068,-504.149,-477.23,-387.50,-566.96,-26.592555
3121,"Ho, Sheng Hui",38:37.79,Singapore University of Social,24.0,4,Open,Run,10000.0,None,S039Z01,...,1884.33,1974.06,2317.79,2317.79,-487.298,-460.379,-433.46,-343.73,-523.19,-24.153572
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19143,"Chow, Justin",43:14.68,Nanyang Polytechnic,17,4,Open,Run,10000.0,None,J962D08,...,1884.33,1974.06,2594.68,2594.68,-764.188,-737.269,-710.35,-620.62,-800.08,-39.582637
19461,"Tan, Bernice",45:02.44,Lacticbuds,25,1,Open,Run,10000.0,None,B075C00,...,2239.86,2346.52,2702.44,2702.44,-526.576,-494.578,-462.58,-355.92,-569.24,-21.684793
19478,"Ho, Sheng Hui",39:03.73,Singapore University of Social,24,8,Open,Run,10000.0,None,S039Z01,...,1884.33,1974.06,2343.73,2343.73,-513.238,-486.319,-459.40,-369.67,-549.13,-25.599019
19508,Shaun Goh,32:20.98,,,16,,"10,000m",,,,...,1884.33,1974.06,1940.98,1940.98,-110.488,-83.569,-56.65,33.08,-146.38,-3.156692


In [153]:
df

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT_x,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,5%,10%,RESULT_CONV,RESULT_BEST,Delta2,Delta3.5,Delta5,Delta10,Delta_Benchmark,PERF_SCALAR
0,SI EN TABITHA NG,11:03.95,,,4.0,,3000m,,,,...,NaN,NaN,663.95,663.95,NaN,NaN,NaN,NaN,NaN,NaN
1,JE AN GARRETT CHUA,6.84,,,3.0,,Long Jump,,,,...,7.1535,6.777,6.84,6.84,-0.5394,-0.42645,-0.3135,0.063,-0.69,-4.163347
2,LAUREL JIA EN LIM,27.71,,,6.0,,Discus Throw,,,,...,46.8730,44.406,27.71,27.71,-20.6432,-19.90310,-19.1630,-16.696,-21.63,-38.838670
3,JOSHUA SHYEN LEE,11.03,,,4.0,,100m,,,,...,10.7730,11.286,11.03,11.03,-0.5648,-0.41090,-0.2570,0.256,-0.77,-2.504873
4,DARYEN XIN TZE KO,54.65,,,2.0,,400m Hurdles,,,,...,53.4555,56.001,54.65,54.65,-2.7218,-1.95815,-1.1945,1.351,-3.74,-2.346297
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20254,"Ng, Zavier",9.88m,Hwa Chong Institution,14,18,U15,Triple Jump,0.0,None,Z854G11,...,15.2855,14.481,9.88,9.88,-5.8882,-5.64685,-5.4055,-4.601,-6.21,-33.595401
20255,"Choo Jia Yi, Allyson",9.30m,Team Start Singapore,17.0,3,Open,Triple Jump,0.0,None,A967D08,...,12.9960,12.312,9.3,9.30,-4.1064,-3.90120,-3.6960,-3.012,-4.38,-27.017544
20256,Lua Yu Xuan,10.02,NYGH,13.5,1.0,C,Triple Jump,,,,...,12.9960,12.312,10.02,10.02,-3.3864,-3.18120,-2.9760,-2.292,-3.66,-21.754386
20257,Muhammad Aaryan Shah Bin Azhar,12.61,SSP,15.5,3.0,B,Triple Jump,,,,...,15.2855,14.481,12.61,12.61,-3.1582,-2.91685,-2.6755,-1.871,-3.48,-16.628341


## Normalize Name Variations

In [154]:
# Fastest execution speed version

import pandas as pd
import re

# Normalize function as before
def normalize_text(s):
    return (str(s)
            .replace('\xa0', '')
            .replace('\r', '')
            .replace('\n', '')
            .strip()
            .casefold())

# Normalize dataframe
df['NAME'] = df['NAME'].apply(normalize_text)

# Load variations file and normalize
file_path = "gs://name_variations/name_variations.csv"
names = pd.read_csv(file_path,
                    sep=',',
                    storage_options={"token": '/Users/veesheenyuen/Desktop/DataScience/Keys/saa-analytics-7c8937b70609.json'})

names['VARIATION'] = names['VARIATION'].apply(normalize_text)
names['NAME'] = names['NAME'].apply(normalize_text)

# Precompile all regex patterns safely

compiled_patterns = []
for pattern_str, replacement in zip(names['VARIATION'], names['NAME']):
    try:
        compiled_re = re.compile(pattern_str)
        compiled_patterns.append( (compiled_re, replacement) )
    except re.error as e:
        print(f"Skipping invalid regex pattern: {pattern_str} Error: {e}")

# Iterate over all patterns and apply replacements using precompiled regexes
for regex, replacement in compiled_patterns:
    df['NAME'] = df['NAME'].str.replace(regex, replacement, regex=True)

# Capitalize final standardized names
df['NAME'] = df['NAME'].str.title()


In [155]:
compiled_patterns

[(re.compile(r'soh, aidan michael zi ren', re.UNICODE),
  'aidan michael soh zi ren'),
 (re.compile(r'aidan michael soh zi ren', re.UNICODE),
  'aidan michael soh zi ren'),
 (re.compile(r'., aidan michael soh zi', re.UNICODE),
  'aidan michael soh zi ren'),
 (re.compile(r'^aidan michael soh$', re.UNICODE), 'aidan michael soh zi ren'),
 (re.compile(r'^ang, chen xiang$', re.UNICODE), 'ang chen xiang'),
 (re.compile(r'^chen xiang ang$', re.UNICODE), 'ang chen xiang'),
 (re.compile(r'^ang, james$', re.UNICODE), 'ang james ethan'),
 (re.compile(r'ang, james ethan ethan', re.UNICODE), 'ang james ethan'),
 (re.compile(r'ang, james ethan ethan kai meng', re.UNICODE),
  'ang james ethan'),
 (re.compile(r'ang, james ethan kai meng', re.UNICODE), 'ang james ethan'),
 (re.compile(r'ang kai meng, james ethan', re.UNICODE), 'ang james ethan'),
 (re.compile(r'james ethan kai meng ang', re.UNICODE), 'ang james ethan'),
 (re.compile(r'ang, james ethan', re.UNICODE), 'ang james ethan'),
 (re.compile(r'^

In [156]:
df

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT_x,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,5%,10%,RESULT_CONV,RESULT_BEST,Delta2,Delta3.5,Delta5,Delta10,Delta_Benchmark,PERF_SCALAR
0,Si En Tabitha Ng,11:03.95,,,4.0,,3000m,,,,...,NaN,NaN,663.95,663.95,NaN,NaN,NaN,NaN,NaN,NaN
1,Je An Chua Garrett,6.84,,,3.0,,Long Jump,,,,...,7.1535,6.777,6.84,6.84,-0.5394,-0.42645,-0.3135,0.063,-0.69,-4.163347
2,Laurel Jia En Lim,27.71,,,6.0,,Discus Throw,,,,...,46.8730,44.406,27.71,27.71,-20.6432,-19.90310,-19.1630,-16.696,-21.63,-38.838670
3,Lee Joshua Shyen,11.03,,,4.0,,100m,,,,...,10.7730,11.286,11.03,11.03,-0.5648,-0.41090,-0.2570,0.256,-0.77,-2.504873
4,Daryen Xin Tze Ko,54.65,,,2.0,,400m Hurdles,,,,...,53.4555,56.001,54.65,54.65,-2.7218,-1.95815,-1.1945,1.351,-3.74,-2.346297
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20254,"Ng, Zavier",9.88m,Hwa Chong Institution,14,18,U15,Triple Jump,0.0,None,Z854G11,...,15.2855,14.481,9.88,9.88,-5.8882,-5.64685,-5.4055,-4.601,-6.21,-33.595401
20255,"Choo Jia Yi, Allyson",9.30m,Team Start Singapore,17.0,3,Open,Triple Jump,0.0,None,A967D08,...,12.9960,12.312,9.3,9.30,-4.1064,-3.90120,-3.6960,-3.012,-4.38,-27.017544
20256,Lua Yu Xuan,10.02,NYGH,13.5,1.0,C,Triple Jump,,,,...,12.9960,12.312,10.02,10.02,-3.3864,-3.18120,-2.9760,-2.292,-3.66,-21.754386
20257,Muhammad Aaryan Shah Bin Azhar,12.61,SSP,15.5,3.0,B,Triple Jump,,,,...,15.2855,14.481,12.61,12.61,-3.1582,-2.91685,-2.6755,-1.871,-3.48,-16.628341


## Remove Foreigners

In [157]:
# Exclude foreigners from MALAYSIA, THAILAND etc.

#df_select = df[(df['TEAM']!='Malaysia') & (df['TEAM']!='THAILAND') & (df['TEAM']!='China') & (df['TEAM']!='South Korea') & (df['TEAM']!='Laos') & (df['TEAM']!='Philippines') & (df['TEAM']!='Piboonbumpen Thailand') & (df['TEAM']!='Chinese Taipei') & (df['TEAM']!='Gurkha Contingent') & (df['TEAM']!='Australia') & (df['TEAM']!='Piboonbumpen Thailand') & (df['TEAM']!='Hong Kong') & (df['TEAM']!='PERAK')] 

df_select = df[(df['TEAM']!='Malaysia')&(df['TEAM']!='THAILAND')&(df['TEAM']!='China')&(df['TEAM']!='Thailand') 
                       &(df['TEAM']!='South Korea')&(df['TEAM']!='Laos')&(df['TEAM']!='Myanmar') 
                       &(df['TEAM']!='Philippines')&(df['TEAM']!='Piboonbumpen Thailand') 
                       &(df['TEAM']!='Chinese Taipei')&(df['TEAM']!='Gurkha Contingent') 
                       &(df['TEAM']!='Australia')&(df['TEAM']!='Piboonbumpen Thailand') 
                       &(df['TEAM']!='Hong Kong')&(df['TEAM']!='PERAK')&(df['TEAM']!='Sri Lanka') 
                       &(df['TEAM']!='Indonesia')&(df['TEAM']!='THAILAND')&(df['TEAM']!='MALAYSIA') 
                       &(df['TEAM']!='PHILIPPINES') & (df['TEAM']!='SOUTH KOREA')&(df['TEAM']!='Waseda') 
                       &(df['TEAM']!='LAOS')&(df['TEAM']!='CHINESE TAIPEI')&(df['TEAM']!='Vietnam')
                       &(df['TEAM']!='INDIA')&(df['TEAM']!='Hong Kong, China')&(df['TEAM']!='AIC JAPAN')
                       &(df['NATIONALITY']!='GBR')&(df['NATIONALITY']!='JPN')&(df['NATIONALITY']!='SRI')&(df['NATIONALITY']!='SAM')
                       &(df['NATIONALITY']!='THA')&(df['NATIONALITY']!='IND')] 

In [158]:
df_select

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT_x,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,5%,10%,RESULT_CONV,RESULT_BEST,Delta2,Delta3.5,Delta5,Delta10,Delta_Benchmark,PERF_SCALAR
0,Si En Tabitha Ng,11:03.95,,,4.0,,3000m,,,,...,NaN,NaN,663.95,663.95,NaN,NaN,NaN,NaN,NaN,NaN
1,Je An Chua Garrett,6.84,,,3.0,,Long Jump,,,,...,7.1535,6.777,6.84,6.84,-0.5394,-0.42645,-0.3135,0.063,-0.69,-4.163347
2,Laurel Jia En Lim,27.71,,,6.0,,Discus Throw,,,,...,46.8730,44.406,27.71,27.71,-20.6432,-19.90310,-19.1630,-16.696,-21.63,-38.838670
3,Lee Joshua Shyen,11.03,,,4.0,,100m,,,,...,10.7730,11.286,11.03,11.03,-0.5648,-0.41090,-0.2570,0.256,-0.77,-2.504873
4,Daryen Xin Tze Ko,54.65,,,2.0,,400m Hurdles,,,,...,53.4555,56.001,54.65,54.65,-2.7218,-1.95815,-1.1945,1.351,-3.74,-2.346297
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20254,"Ng, Zavier",9.88m,Hwa Chong Institution,14,18,U15,Triple Jump,0.0,None,Z854G11,...,15.2855,14.481,9.88,9.88,-5.8882,-5.64685,-5.4055,-4.601,-6.21,-33.595401
20255,"Choo Jia Yi, Allyson",9.30m,Team Start Singapore,17.0,3,Open,Triple Jump,0.0,None,A967D08,...,12.9960,12.312,9.3,9.30,-4.1064,-3.90120,-3.6960,-3.012,-4.38,-27.017544
20256,Lua Yu Xuan,10.02,NYGH,13.5,1.0,C,Triple Jump,,,,...,12.9960,12.312,10.02,10.02,-3.3864,-3.18120,-2.9760,-2.292,-3.66,-21.754386
20257,Muhammad Aaryan Shah Bin Azhar,12.61,SSP,15.5,3.0,B,Triple Jump,,,,...,15.2855,14.481,12.61,12.61,-3.1582,-2.91685,-2.6755,-1.871,-3.48,-16.628341


In [159]:
df_select[df_select['NAME']=='LEE, VANESSA']

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT_x,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,5%,10%,RESULT_CONV,RESULT_BEST,Delta2,Delta3.5,Delta5,Delta10,Delta_Benchmark,PERF_SCALAR


In [160]:
# Read list of foreigners from GCS bucket

file_path = "gs://name_lists/List of Foreigners.csv"
foreigners = pd.read_csv(file_path,
                 sep=",",
                 encoding="unicode escape",
                 storage_options={"token": '/Users/veesheenyuen/Desktop/DataScience/Keys/saa-analytics-7c8937b70609.json'})


In [161]:
foreigners

,LAST_NAME,FIRST_NAME
0,Aaryan,Greuter Christoph
1,Akahodani,Takayuki
2,Apondar,Audric
3,Brooks,Ruby
4,Brouwer,Cees
...,...,...
235,Kashama,Biwesa Daniel
236,ISMAIL,MUHAMMAD ZULFIQAR
237,Jayaganeson,Kirtisha
238,LIN,Yu Sian


In [162]:
df_select['NAME'].str.casefold()

0                      si en tabitha ng
1                    je an chua garrett
2                     laurel jia en lim
3                      lee joshua shyen
4                     daryen xin tze ko
                      ...              
20254                        ng, zavier
20255              choo jia yi, allyson
20256                       lua yu xuan
20257    muhammad aaryan shah bin azhar
20258                   carrie-anne loh
Name: NAME, Length: 19988, dtype: object

In [163]:
foreigners['V1'] = foreigners['LAST_NAME']+' '+foreigners['FIRST_NAME']
foreigners['V2'] = foreigners['FIRST_NAME']+' '+foreigners['LAST_NAME']
foreigners['V3'] = foreigners['LAST_NAME']+', '+foreigners['FIRST_NAME']
foreigners['V4'] = foreigners['FIRST_NAME']+' '+foreigners['LAST_NAME']

for1 = foreigners['V1'].dropna().tolist()
for2 = foreigners['V2'].dropna().tolist()
for3 = foreigners['V3'].dropna().tolist()
for4 = foreigners['V4'].dropna().tolist()

foreign_list = for1+for2+for3+for4 

foreign_list_casefold=[s.casefold() for s in foreign_list]

exclusions = foreign_list_casefold

no_foreigners_list = df_select.loc[~df['NAME'].str.casefold().isin(exclusions)]  # ~ means NOT IN. DROP spex carded athletes

In [164]:
no_foreigners_list

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT_x,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,5%,10%,RESULT_CONV,RESULT_BEST,Delta2,Delta3.5,Delta5,Delta10,Delta_Benchmark,PERF_SCALAR
0,Si En Tabitha Ng,11:03.95,,,4.0,,3000m,,,,...,NaN,NaN,663.95,663.95,NaN,NaN,NaN,NaN,NaN,NaN
1,Je An Chua Garrett,6.84,,,3.0,,Long Jump,,,,...,7.1535,6.777,6.84,6.84,-0.5394,-0.42645,-0.3135,0.063,-0.69,-4.163347
2,Laurel Jia En Lim,27.71,,,6.0,,Discus Throw,,,,...,46.8730,44.406,27.71,27.71,-20.6432,-19.90310,-19.1630,-16.696,-21.63,-38.838670
3,Lee Joshua Shyen,11.03,,,4.0,,100m,,,,...,10.7730,11.286,11.03,11.03,-0.5648,-0.41090,-0.2570,0.256,-0.77,-2.504873
4,Daryen Xin Tze Ko,54.65,,,2.0,,400m Hurdles,,,,...,53.4555,56.001,54.65,54.65,-2.7218,-1.95815,-1.1945,1.351,-3.74,-2.346297
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20254,"Ng, Zavier",9.88m,Hwa Chong Institution,14,18,U15,Triple Jump,0.0,None,Z854G11,...,15.2855,14.481,9.88,9.88,-5.8882,-5.64685,-5.4055,-4.601,-6.21,-33.595401
20255,"Choo Jia Yi, Allyson",9.30m,Team Start Singapore,17.0,3,Open,Triple Jump,0.0,None,A967D08,...,12.9960,12.312,9.3,9.30,-4.1064,-3.90120,-3.6960,-3.012,-4.38,-27.017544
20256,Lua Yu Xuan,10.02,NYGH,13.5,1.0,C,Triple Jump,,,,...,12.9960,12.312,10.02,10.02,-3.3864,-3.18120,-2.9760,-2.292,-3.66,-21.754386
20257,Muhammad Aaryan Shah Bin Azhar,12.61,SSP,15.5,3.0,B,Triple Jump,,,,...,15.2855,14.481,12.61,12.61,-3.1582,-2.91685,-2.6755,-1.871,-3.48,-16.628341


## Athlete Selection

In [165]:
# Choose best performance per event and athlete (robust version)

# 1) Convert to numeric FIRST
no_foreigners_list["PERF_SCALAR"] = pd.to_numeric(no_foreigners_list["PERF_SCALAR"], errors="coerce")

# 2) Drop non-numeric (NaNs) and non-finite values
df2 = no_foreigners_list[np.isfinite(no_foreigners_list["PERF_SCALAR"])].copy()

# 3) Normalize grouping keys to avoid silent split by stray spaces
for col in ["NAME", "MAPPED_EVENT"]:
    df2[col] = df2[col].astype(str).str.strip()

# 4) Keep the best (largest) per group
top_performers_clean = (
    df2.sort_values(
        ["MAPPED_EVENT", "NAME", "PERF_SCALAR"],
        ascending=[True, True, False]
    )
    .drop_duplicates(subset=["MAPPED_EVENT", "NAME"], keep="first")
    .reset_index(drop=True)
)

top_performers_clean

/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_69858/3454415374.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  no_foreigners_list["PERF_SCALAR"] = pd.to_numeric(no_foreigners_list["PERF_SCALAR"], errors="coerce")


,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT_x,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,5%,10%,RESULT_CONV,RESULT_BEST,Delta2,Delta3.5,Delta5,Delta10,Delta_Benchmark,PERF_SCALAR
0,"Ahmed, Nawaz",39:49.85,Lacticbuds,22,11,Open,Run,10000.0,None,N034I03,...,1884.3300,1974.060,2389.85,2389.85,-559.3580,-532.43900,-505.5200,-415.790,-595.25,-28.168951
1,"Bin Abdul Rashid, Ar Rizqi",41:53.77,Cougars Athletic Association,19,10,Open,Run,10000.0,None,A913D06,...,1884.3300,1974.060,2513.77,2513.77,-683.2780,-656.35900,-629.4400,-539.710,-719.17,-35.074111
2,"Branson, Kwong Chen Jun",42:55.64,Nanyang Polytechnic,17,3,Open,Run,10000.0,None,K490D08,...,1884.3300,1974.060,2575.64,2575.64,-745.1480,-718.22900,-691.3100,-601.580,-781.04,-38.521676
3,"Chai, Wen Bin",35:15.52,Singapore Institute of Technol,0.0,5,Open,Run,10000.0,None,None,...,1884.3300,1974.060,2115.52,2115.52,-285.0280,-258.10900,-231.1900,-141.460,-320.92,-12.882536
4,"Chan, Yao Li",40:17.10,Republic Polytechnic,23,9,Open,Run,10000.0,None,Y065F02,...,1884.3300,1974.060,2417.1,2417.10,-586.6080,-559.68900,-532.7700,-443.040,-622.50,-29.687396
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10296,Zhao Daniel,11.98,-,,1,,Triple Jump,,nan,,...,15.2855,14.481,11.98,11.98,-3.7882,-3.54685,-3.3055,-2.501,-4.11,-20.543816
10297,"Zhao, Daniel",11.67m,Hwa Chong Institution,14,3,U15,Triple Jump,0.0,None,None,...,15.2855,14.481,11.67,11.67,-4.0982,-3.85685,-3.6155,-2.811,-4.42,-22.470479
10298,Zheng Justin De,10.47m,National Junior College,14,12,U15,Triple Jump,0.0,None,J034B11,...,15.2855,14.481,10.47,10.47,-5.2982,-5.05685,-4.8155,-4.011,-5.62,-29.928527
10299,Zhong Chuhan,12.26,,,,,Triple Jump,,,,...,12.9960,12.312,12.26,12.26,-1.1464,-0.94120,-0.7360,-0.052,-1.42,-5.380117


In [166]:
top_performers_clean.reset_index(inplace=True)


In [167]:
top_performers_clean

,index,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT_x,DISTANCE,EVENT_CLASS,...,5%,10%,RESULT_CONV,RESULT_BEST,Delta2,Delta3.5,Delta5,Delta10,Delta_Benchmark,PERF_SCALAR
0,0,"Ahmed, Nawaz",39:49.85,Lacticbuds,22,11,Open,Run,10000.0,None,...,1884.3300,1974.060,2389.85,2389.85,-559.3580,-532.43900,-505.5200,-415.790,-595.25,-28.168951
1,1,"Bin Abdul Rashid, Ar Rizqi",41:53.77,Cougars Athletic Association,19,10,Open,Run,10000.0,None,...,1884.3300,1974.060,2513.77,2513.77,-683.2780,-656.35900,-629.4400,-539.710,-719.17,-35.074111
2,2,"Branson, Kwong Chen Jun",42:55.64,Nanyang Polytechnic,17,3,Open,Run,10000.0,None,...,1884.3300,1974.060,2575.64,2575.64,-745.1480,-718.22900,-691.3100,-601.580,-781.04,-38.521676
3,3,"Chai, Wen Bin",35:15.52,Singapore Institute of Technol,0.0,5,Open,Run,10000.0,None,...,1884.3300,1974.060,2115.52,2115.52,-285.0280,-258.10900,-231.1900,-141.460,-320.92,-12.882536
4,4,"Chan, Yao Li",40:17.10,Republic Polytechnic,23,9,Open,Run,10000.0,None,...,1884.3300,1974.060,2417.1,2417.10,-586.6080,-559.68900,-532.7700,-443.040,-622.50,-29.687396
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10296,10296,Zhao Daniel,11.98,-,,1,,Triple Jump,,nan,...,15.2855,14.481,11.98,11.98,-3.7882,-3.54685,-3.3055,-2.501,-4.11,-20.543816
10297,10297,"Zhao, Daniel",11.67m,Hwa Chong Institution,14,3,U15,Triple Jump,0.0,None,...,15.2855,14.481,11.67,11.67,-4.0982,-3.85685,-3.6155,-2.811,-4.42,-22.470479
10298,10298,Zheng Justin De,10.47m,National Junior College,14,12,U15,Triple Jump,0.0,None,...,15.2855,14.481,10.47,10.47,-5.2982,-5.05685,-4.8155,-4.011,-5.62,-29.928527
10299,10299,Zhong Chuhan,12.26,,,,,Triple Jump,,,...,12.9960,12.312,12.26,12.26,-1.1464,-0.94120,-0.7360,-0.052,-1.42,-5.380117


In [168]:
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/SEAG_u18/')

top_performers_clean.to_csv('seag_u18_top_performers.csv', encoding='utf-8')

In [219]:
# Choose best performance for each event

#tiered_performers = top_performers_clean.sort_values(['GENDER', 'MAPPED_EVENT', 'PERF_SCALAR'],ascending=False).groupby(['MAPPED_EVENT', 'NAME']).head(1)

tiered_performers = top_performers_clean


In [220]:
tiered_performers

,index,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT_x,DISTANCE,EVENT_CLASS,...,10%,RESULT_CONV,RESULT_BEST,Delta2,Delta3.5,Delta5,Delta10,Delta_Benchmark,PERF_SCALAR,TIER
0,0,"Ahmed, Nawaz",39:49.85,Lacticbuds,22,11,Open,Run,10000.0,None,...,1974.060,2389.85,2389.85,-559.3580,-532.43900,-505.5200,-415.790,-595.25,-28.168951,
1,1,"Bin Abdul Rashid, Ar Rizqi",41:53.77,Cougars Athletic Association,19,10,Open,Run,10000.0,None,...,1974.060,2513.77,2513.77,-683.2780,-656.35900,-629.4400,-539.710,-719.17,-35.074111,
2,2,"Branson, Kwong Chen Jun",42:55.64,Nanyang Polytechnic,17,3,Open,Run,10000.0,None,...,1974.060,2575.64,2575.64,-745.1480,-718.22900,-691.3100,-601.580,-781.04,-38.521676,
3,3,"Chai, Wen Bin",35:15.52,Singapore Institute of Technol,0.0,5,Open,Run,10000.0,None,...,1974.060,2115.52,2115.52,-285.0280,-258.10900,-231.1900,-141.460,-320.92,-12.882536,
4,4,"Chan, Yao Li",40:17.10,Republic Polytechnic,23,9,Open,Run,10000.0,None,...,1974.060,2417.1,2417.10,-586.6080,-559.68900,-532.7700,-443.040,-622.50,-29.687396,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10296,10296,Zhao Daniel,11.98,-,,1,,Triple Jump,,nan,...,14.481,11.98,11.98,-3.7882,-3.54685,-3.3055,-2.501,-4.11,-20.543816,
10297,10297,"Zhao, Daniel",11.67m,Hwa Chong Institution,14,3,U15,Triple Jump,0.0,None,...,14.481,11.67,11.67,-4.0982,-3.85685,-3.6155,-2.811,-4.42,-22.470479,
10298,10298,Zheng Justin De,10.47m,National Junior College,14,12,U15,Triple Jump,0.0,None,...,14.481,10.47,10.47,-5.2982,-5.05685,-4.8155,-4.011,-5.62,-29.928527,
10299,10299,Zhong Chuhan,12.26,,,,,Triple Jump,,,...,12.312,12.26,12.26,-1.1464,-0.94120,-0.7360,-0.052,-1.42,-5.380117,


## Apply tiering of performance rules

In [221]:
# Identify Tier 1/2/3 performers

#top_performers_clean['TIER'] = np.where((top_performers_clean['Delta_Benchmark']>=0), 'Tier 1',    
#                                np.where(((top_performers_clean['Delta_Benchmark']<0) & (top_performers_clean['Delta2']>=0)), 'Tier2',
#                                np.where(((top_performers_clean['Delta2']<0) & (top_performers_clean['Delta3.5']>=0)), 'Tier3', ' ')))


tiered_performers['TIER'] = np.where((tiered_performers['Delta_Benchmark']>=0), 'Tier 1',    
                                np.where(((tiered_performers['Delta_Benchmark']<0) & (tiered_performers['Delta2']>=0)), 'Tier 2',
                                np.where(((tiered_performers['Delta2']<0) & (tiered_performers['Delta3.5']>=0)), 'Tier 3',
                                np.where(((tiered_performers['Delta3.5']<0) & (tiered_performers['Delta5']>=0)), 'Tier 4',
                                np.where(((tiered_performers['Delta5']<0) & (tiered_performers['Delta10']>=0)), 'Tier 5', ' ')))))



In [222]:
tiered_performers[tiered_performers['MAPPED_EVENT']=='5000m']

,index,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT_x,DISTANCE,EVENT_CLASS,...,10%,RESULT_CONV,RESULT_BEST,Delta2,Delta3.5,Delta5,Delta10,Delta_Benchmark,PERF_SCALAR,TIER
6026,6026,"Adarsh, Aravinth",15:44.09,National University Singapore,24.0,9,Open,Run,5000.0,None,...,976.8,944.09,944.09,-38.33,-25.01,-11.69,32.71,-56.09,-1.316441,Tier 5
6027,6027,"Ahmed, Nawaz",17:19.53,Lacticbuds,22,7,Open,Run,5000.0,None,...,976.8,1039.53,1039.53,-133.77,-120.45,-107.13,-62.73,-151.53,-12.064189,
6028,6028,"Aravinth, Adarsh",16:07.40,National University Singapore,0.0,1,Open,Run,5000.0,None,...,976.8,967.4,967.40,-61.64,-48.32,-35.00,9.40,-79.40,-3.941441,Tier 5
6029,6029,Ayden Tan Chee Yew,17:47.35,VJC,17.5,7.0,A,5000m,,,...,976.8,1067.35,1067.35,-161.59,-148.27,-134.95,-90.55,-179.35,-15.197072,
6030,6030,"Bin Abdul Rashid, Ar Rizqi",18:17.21,Cougars Athletic Association,19,8,Open,Run,5000.0,None,...,976.8,1097.21,1097.21,-191.45,-178.13,-164.81,-120.41,-209.21,-18.559685,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6163,6163,"Yip, Tony",19:50.47,Singapore Institute of Managem,24,20,Open,Run,5000.0,None,...,976.8,1190.47,1190.47,-284.71,-271.39,-258.07,-213.67,-302.47,-29.061937,
6164,6164,"Yip, Wan Hoi",20:19.51,Erovra Club,25,24,Open,Run,5000.0,None,...,976.8,1219.51,1219.51,-313.75,-300.43,-287.11,-242.71,-331.51,-32.332207,
6165,6165,"Zhi Han, Lim",19:34.97,TeamFabian,18,11,Open,Run,5000.0,None,...,976.8,1174.97,1174.97,-269.21,-255.89,-242.57,-198.17,-286.97,-27.316441,
6166,6166,"Zhi Yan, Lim",20:44.53,Oldham Athletics,22,27,Open,Run,5000.0,None,...,976.8,1244.53,1244.53,-338.77,-325.45,-312.13,-267.73,-356.53,-35.149775,


In [237]:
# Drop rows without a SEAG benchmark

final_df = tiered_performers[tiered_performers['BENCHMARK'].notna()]


In [238]:
final_df

,index,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT_x,DISTANCE,EVENT_CLASS,...,10%,RESULT_CONV,RESULT_BEST,Delta2,Delta3.5,Delta5,Delta10,Delta_Benchmark,PERF_SCALAR,TIER
0,0,"Ahmed, Nawaz",39:49.85,Lacticbuds,22,11,Open,Run,10000.0,None,...,1974.060,2389.85,2389.85,-559.3580,-532.43900,-505.5200,-415.790,-595.25,-28.168951,
1,1,"Bin Abdul Rashid, Ar Rizqi",41:53.77,Cougars Athletic Association,19,10,Open,Run,10000.0,None,...,1974.060,2513.77,2513.77,-683.2780,-656.35900,-629.4400,-539.710,-719.17,-35.074111,
2,2,"Branson, Kwong Chen Jun",42:55.64,Nanyang Polytechnic,17,3,Open,Run,10000.0,None,...,1974.060,2575.64,2575.64,-745.1480,-718.22900,-691.3100,-601.580,-781.04,-38.521676,
3,3,"Chai, Wen Bin",35:15.52,Singapore Institute of Technol,0.0,5,Open,Run,10000.0,None,...,1974.060,2115.52,2115.52,-285.0280,-258.10900,-231.1900,-141.460,-320.92,-12.882536,
4,4,"Chan, Yao Li",40:17.10,Republic Polytechnic,23,9,Open,Run,10000.0,None,...,1974.060,2417.1,2417.10,-586.6080,-559.68900,-532.7700,-443.040,-622.50,-29.687396,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10296,10296,Zhao Daniel,11.98,-,,1,,Triple Jump,,nan,...,14.481,11.98,11.98,-3.7882,-3.54685,-3.3055,-2.501,-4.11,-20.543816,
10297,10297,"Zhao, Daniel",11.67m,Hwa Chong Institution,14,3,U15,Triple Jump,0.0,None,...,14.481,11.67,11.67,-4.0982,-3.85685,-3.6155,-2.811,-4.42,-22.470479,
10298,10298,Zheng Justin De,10.47m,National Junior College,14,12,U15,Triple Jump,0.0,None,...,14.481,10.47,10.47,-5.2982,-5.05685,-4.8155,-4.011,-5.62,-29.928527,
10299,10299,Zhong Chuhan,12.26,,,,,Triple Jump,,,...,12.312,12.26,12.26,-1.1464,-0.94120,-0.7360,-0.052,-1.42,-5.380117,


In [239]:
final_tiered_selection = final_df[final_df['TIER']!=' ']

In [240]:
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/SEAG_u18/')

final_tiered_selection.to_csv('seag_u18_tiered_performers.csv', encoding='utf-8')

## Determine Age of Athletes

In [241]:
final_tiered_selection["DOB"] = pd.to_datetime(final_tiered_selection["DOB"], errors="coerce", dayfirst=True)


/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_69858/505975541.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_tiered_selection["DOB"] = pd.to_datetime(final_tiered_selection["DOB"], errors="coerce", dayfirst=True)


In [242]:
# Create a list of name components to seach for permutations

final_tiered_selection['name_to_list'] = final_tiered_selection['clean_name'].str.split(' ')

final_tiered_selection

/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_69858/3643911941.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_tiered_selection['name_to_list'] = final_tiered_selection['clean_name'].str.split(' ')


,index,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT_x,DISTANCE,EVENT_CLASS,...,RESULT_CONV,RESULT_BEST,Delta2,Delta3.5,Delta5,Delta10,Delta_Benchmark,PERF_SCALAR,TIER,name_to_list
14,14,Goh Shaun,31:02.40,,,5,,"10,000m",,,...,1862.4,1862.40,-31.9080,-4.98900,21.9300,111.660,-67.80,1.221999,Tier 4,"[shaun, goh]"
15,15,Goh Shing Ling,38:30.59,TeamFabian,26,1,Open,Run,10000.0,None,...,2310.59,2310.59,-134.7260,-102.72800,-70.7300,35.930,-177.39,-3.315676,Tier 5,"[shing, ling, goh]"
16,16,"Goh, Shaun",31:02.40,,28.0,5,OPEN,"10,000m",,,...,1862.4,1862.40,-31.9080,-4.98900,21.9300,111.660,-67.80,1.221999,Tier 4,"[goh, shaun]"
26,26,Lee Vanessa,36:15.67,,27.0,1,OPEN,"10,000m",,,...,2175.67,2175.67,0.1940,32.19200,64.1900,170.850,-42.47,3.009094,Tier 2,"[lee, vanessa]"
45,45,Soh Rui Yong Guillaume,31:31.91,<NA>,<NA>,7.0,<NA>,"10,000m",<NA>,<NA>,...,1891.91,1891.91,-61.4180,-34.49900,-7.5800,82.150,-97.31,-0.422378,Tier 5,"[rui, yong, soh]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10107,10107,Lee Gabriel Jin Yi,16.09,<NA>,<NA>,3.0,<NA>,Triple Jump,<NA>,<NA>,...,16.09,16.09,0.3218,0.56315,0.8045,1.609,0.00,5.000000,Tier 1,"[gabriel, lee]"
10145,10145,Medina Andrew George,14.98m,National University Singapore,23,1,Open,Triple Jump,0.0,None,...,14.98,14.98,-0.7882,-0.54685,-0.3055,0.499,-1.11,-1.898695,Tier 5,"[medina, andrew]"
10196,10196,Rozario Tia Louise,13.00,<NA>,<NA>,5.0,<NA>,Triple Jump,<NA>,<NA>,...,13.0,13.00,-0.4064,-0.20120,0.0040,0.688,-0.68,0.029240,Tier 4,"[tia, louise, rozario]"
10209,10209,Shou Yi Rei Tan,14.49,,,4.0,,Triple Jump,,,...,14.49,14.49,-1.2782,-1.03685,-0.7955,0.009,-1.60,-4.944065,Tier 5,"[shou, yi, rei, tan]"


In [243]:
import itertools
from itertools import permutations

def safe_permutations_limited(name_components, max_perms=1000, max_name_len=6):
    """
    Generate limited permutations with strict safety limits.
    """
    if not isinstance(name_components, (list, tuple)):
        return []
    
    # SAFETY 1: Skip overly long names
    if len(name_components) > max_name_len:
        return []
    
    # SAFETY 2: Calculate total permutations first
    total_perms = 1
    for i in range(1, len(name_components) + 1):
        total_perms *= i
    
    # SAFETY 3: Skip if too many permutations
    if total_perms > max_perms * 2:  # Conservative threshold
        return []
    
    # Generate and immediately limit
    all_perms = list(permutations(name_components))
    
    # SAFETY 4: Hard truncate
    if len(all_perms) > max_perms:
        all_perms = all_perms[:max_perms]
    
    return [" ".join(p) for p in all_perms]

def generate_permutations_safe(df, name_col='name_to_list', dob_dict=None, max_perms=500, max_name_len=5):
    """
    Safe version with permutation limits.
    """
    if dob_dict is None:
        dob_dict = dictionary_dob_clean_name
    
    df['permutations'] = [[] for _ in range(len(df))]
    df['dob_list'] = [[] for _ in range(len(df))]
    
    for idx in df.index:
        name_components = df.at[idx, name_col]
        
        perms_list = safe_permutations_limited(
            name_components, 
            max_perms=max_perms,
            max_name_len=max_name_len  # Now passed correctly
        )
        
        if not perms_list:
            continue
        
        dob_list = [dob_dict.get(p) for p in perms_list if dob_dict.get(p) is not None]
        
        df.at[idx, 'permutations'] = perms_list
        df.at[idx, 'dob_list'] = dob_list
        
        if idx % 1000 == 0:
            print(f"Processed {idx}/{len(df)} rows")
    
    return df



# Usage - SAFE
final_tiered_selection = generate_permutations_safe(
    final_tiered_selection,
    max_perms=500,      # Max 500 permutations per athlete
    max_name_len=5      # Skip names with >5 components
)


/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_69858/3897168578.py:40: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['permutations'] = [[] for _ in range(len(df))]
/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_69858/3897168578.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['dob_list'] = [[] for _ in range(len(df))]


In [244]:
# Create a new column with the first element of dob_list (or NaN if empty)
final_tiered_selection['dob_first'] = final_tiered_selection['dob_list'].apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None)


/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_69858/3859836490.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_tiered_selection['dob_first'] = final_tiered_selection['dob_list'].apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None)


In [245]:
# Choose dob_parsed as priority unless dob_parsed is empty

final_tiered_selection['DOB_parsed'] = final_tiered_selection['DOB_parsed'].replace('None', np.nan)


# Correct logic: col1 if exists, else col2
final_tiered_selection['dob_stage1'] = np.where(
    final_tiered_selection['DOB_parsed'].notna(), 
    final_tiered_selection['DOB_parsed'], 
    final_tiered_selection['dob_first']
)




/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_69858/2480854039.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_tiered_selection['DOB_parsed'] = final_tiered_selection['DOB_parsed'].replace('None', np.nan)
/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_69858/2480854039.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_tiered_selection['dob_stage1'] = np.where(


In [247]:
# If NSG event then choose AGE otherwise choose age_extract

condition1 = final_tiered_selection['COMPETITION']=='National School Games'

final_tiered_selection['dob_final'] = final_tiered_selection['AGE'].where((condition1), final_tiered_selection['dob_stage1'].values)


/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_69858/4163669693.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_tiered_selection['dob_final'] = final_tiered_selection['AGE'].where((condition1), final_tiered_selection['dob_stage1'].values)


In [248]:
# Determine exact age of athletes as at Dec 31 2025
# If athletes has no null value for age then he/she will be filtered out hence check for missing DOBs

end = pd.Timestamp('2025-12-31')

# 2) Force DOB column to naive datetime (cast to string first to strip any tz info)
dob = pd.to_datetime(final_tiered_selection['dob_final'].astype(str), errors='coerce')

# 3) Compute age in years (days / 365.25)
final_tiered_selection['age_dec25'] = (end - dob).dt.days / 365.25

# 4) Filter: >19 and <20 years old on 2025-12-31
#mask = (age_years > 19) & (age_years < 20)
#athletes = athletes.loc[mask].copy()
#athletes['age_years'] = age_years[mask]


/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_69858/2369437300.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_tiered_selection['age_dec25'] = (end - dob).dt.days / 365.25


In [250]:
final_tiered_selection[['age_dec25']] = final_tiered_selection[['age_dec25']].apply(pd.to_numeric)

mask = (((final_tiered_selection['age_dec25'] <= 18))|(final_tiered_selection['age_dec25'].isna()))
#mask = ((final_df['age_extract'].isnull()))


final_tiered_selection = final_tiered_selection.loc[mask]


/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_69858/2286613888.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_tiered_selection[['age_dec25']] = final_tiered_selection[['age_dec25']].apply(pd.to_numeric)


In [251]:
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/SEAG_u18/')

final_tiered_selection.to_csv('post_dob_map.csv', encoding='utf-8')

In [745]:
final_df['DOB_new'] = pd.to_datetime(final_df['DOB_new'], errors='coerce')

final_df['year_extract']=final_df['DOB_new'].dt.strftime('%Y')

final_df['year_extract'] = pd.to_numeric(final_df['year_extract'])

final_df['age_extract'] = 2025 - final_df['year_extract']


In [746]:
def age(number):  # correct negative age numbers
    
    if number<0:
        
        number+=100
        
    return number


final_df['age_extract']=final_df['age_extract'].apply(age)


In [747]:
# If NSG event then choose AGE otherwise choose age_extract

condition1 = final_df['COMPETITION']=='National School Games'
#condition2=((df['CATEGORY_EVENT']=='Jump')|(df['CATEGORY_EVENT']=='Throw'))
#condition3=df['SEED_CONV']<df['RESULT_CONV']
#condition4=~((df['CATEGORY_EVENT']=='Jump')|(df['CATEGORY_EVENT']=='Throw'))


final_df['age_extract'] = final_df['AGE'].where((condition1), final_df['age_extract'].values)


In [748]:
# Change to numeric

final_df[['age_extract']] = final_df[['age_extract']].apply(pd.to_numeric)

In [752]:
# Convert time format for marathon and 5000m into mm:ss.00
# Choose the correct column indices or you will get erratic timings

import datetime

#s=247.779

#datetime.datetime.fromtimestamp(s).strftime('%M:%S.%f')

all_ranking=final_tiered_selection.reset_index(drop=True)


#all_ranking[['2%', '3.5%', '5%']] = df[['2%', '3.5%', '5%']].apply(pd.to_numeric)


#all_ranking['2%'] = all_ranking['2%'].astype("string")
#all_ranking['3.5%'] = all_ranking['3.5%'].astype("string")
#all_ranking['5%'] = all_ranking['5%'].astype("string")


for i in range(len(all_ranking)):
        
    rowIndex = all_ranking.index[i]

    event=all_ranking.loc[rowIndex,'MAPPED_EVENT']
        
    
#    time_base2=all_ranking.iloc[rowIndex,25]
#    time_base3=all_ranking.iloc[rowIndex,26]
#    time_base5=all_ranking.iloc[rowIndex,27]
#    time_base10=all_ranking.iloc[rowIndex,27]
    time_base2=all_ranking.loc[rowIndex,'2%']
    time_base3=all_ranking.loc[rowIndex,'3.5%']
    time_base5=all_ranking.loc[rowIndex,'5%']
    time_base10=all_ranking.loc[rowIndex,'10%']
    
        
    if all_ranking.loc[rowIndex,'Metric']==None:
        continue
        
    if event=='800m' or event=='10,000m' or event=='5000m' or event=='3000m Steeplechase' or event=='1500m':
        
      #  print(i, event, time_base2, time_base3, time_base5)
   
        
        date_preconvert2 = datetime.datetime.utcfromtimestamp(time_base2)
        date_preconvert3 = datetime.datetime.utcfromtimestamp(time_base3)
        date_preconvert5 = datetime.datetime.utcfromtimestamp(time_base5)
        date_preconvert10 = datetime.datetime.utcfromtimestamp(time_base10)

        
    #    print(date_preconvert2, date_preconvert3, date_preconvert5)
            
        
        output2 = datetime.datetime.strftime(date_preconvert2, "%M:%S.%f")
        output3 = datetime.datetime.strftime(date_preconvert3, "%M:%S.%f")
        output5 = datetime.datetime.strftime(date_preconvert5, "%M:%S.%f")
        output10 = datetime.datetime.strftime(date_preconvert10, "%M:%S.%f")

        
     #   print(event, output2, output3, output5)

                    
       #     top_performers_clean.loc[rowIndex, '2%_timing'] = output2
       #     top_performers_clean.loc[rowIndex, '3.5%_timing'] = output3
       #     top_performers_clean.loc[rowIndex, '5%_timing'] = output5
            
   
        all_ranking.at[rowIndex, '2%'] = output2 # copy over time format
        all_ranking.at[rowIndex, '3.5%'] = output3
        all_ranking.at[rowIndex, '5%'] = output5
        all_ranking.at[rowIndex, '10%'] = output10

        
    elif event=='Marathon':
        
      #  print(time_base2, time_base3, time_base5)

        
        try:
            

        
            date_preconvert2 = datetime.datetime.utcfromtimestamp(time_base2)
            date_preconvert3 = datetime.datetime.utcfromtimestamp(time_base3)
            date_preconvert5 = datetime.datetime.utcfromtimestamp(time_base5)
            date_preconvert10 = datetime.datetime.utcfromtimestamp(time_base10)

            
            
            output2 = datetime.datetime.strftime(date_preconvert2, "%H:%M:%S")
            output3 = datetime.datetime.strftime(date_preconvert3, "%H:%M:%S")
            output5 = datetime.datetime.strftime(date_preconvert5, "%H:%M:%S")
            output10 = datetime.datetime.strftime(date_preconvert10, "%H:%M:%S")

            
        
        #    top_performers_clean.loc[rowIndex, '2%_timing'] = output2
        #    top_performers_clean.loc[rowIndex, '3.5%_timing'] = output3
        #    top_performers_clean.loc[rowIndex, '5%_timing'] = output5
            
            all_ranking.at[rowIndex, '2%'] = output2 # copy over time format
            all_ranking.at[rowIndex, '3.5%'] = output3
            all_ranking.at[rowIndex, '5%'] = output5
            all_ranking.at[rowIndex, '10%'] = output10

            
         #   print('output', output2, output3, output5)


        
        except:
            
            pass
                        
             


/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_93740/405696043.py:69: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '32:03.210000' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  all_ranking.at[rowIndex, '2%'] = output2 # copy over time format
/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_93740/405696043.py:70: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '32:31.492500' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  all_ranking.at[rowIndex, '3.5%'] = output3
/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_93740/405696043.py:71: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '32:59.775000' has dtype incompatible with float64

In [753]:
all_ranking

,index,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT_x,DISTANCE,EVENT_CLASS,...,Delta2,Delta3.5,Delta5,Delta10,Delta_Benchmark,PERF_SCALAR,TIER,DOB_new,year_extract,age_extract
0,9,Goh Shaun,31:02.40,,,5,,"10,000m",,,...,60.8100,89.0925,117.375,211.650,23.10,6.225139,Tier 1,1997-12-01,1997.0,28.0
1,10,Goh Shing Ling,38:44.43,TeamFabian,12,1,Open,Run,10000.0,None,...,-150.8100,-118.8450,-86.880,19.670,-193.43,-4.076959,Tier 5,1999-07-06,1999.0,26.0
2,11,"Goh, Shaun",31:02.40,,12,5,OPEN,"10,000m",,,...,60.8100,89.0925,117.375,211.650,23.10,6.225139,Tier 1,NaT,NaN,NaN
3,18,Ko Wen Qiang Keane,33:51.89,Club ZOOM,12,1,Open,Run,10000.0,None,...,-108.6800,-80.3975,-52.115,42.160,-146.39,-2.763988,Tier 5,2000-07-26,2000.0,25.0
4,19,Lee Vanessa,36:15.67,,12,1,OPEN,"10,000m",,,...,-2.0500,29.9150,61.880,168.430,-44.67,2.903801,Tier 3,NaT,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
473,8683,Rozario Tia Louise,12.90m,FAC,12,1,Open,Triple Jump,0.0,None,...,-0.2908,-0.0889,0.113,0.786,-0.56,0.839525,Tier 4,2000-10-14,2000.0,25.0
474,8708,Tan Shou Ri Yei (Chen Shouyi),14.31,RI,18.5,1.0,A,Triple Jump,,,...,-1.0760,-0.8405,-0.605,0.180,-1.39,-3.853503,Tier 5,NaT,NaN,18.5
475,8709,Tan Shou Yi Rei,14.99m,Raffles Institution JC,17,1,U20,Triple Jump,0.0,None,...,-0.3960,-0.1605,0.075,0.860,-0.71,0.477707,Tier 4,2008-05-12,2008.0,17.0
476,8732,Tang Kai Sheng Cayman,14.17m,VICTORIA SCHOOL,16,1,U18,Triple Jump,0.0,None,...,-1.2160,-0.9805,-0.745,0.040,-1.53,-4.745223,Tier 5,2009-07-15,2009.0,16.0


In [754]:
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/SEAG_u18/')

all_ranking.to_csv('all_ranking_seag_u18.csv', encoding='utf-8')

# End of Initial Selection

In [108]:
# Rank everyone for published ranking lists

published_ranking = final_df.sort_values(['MAPPED_EVENT','GENDER','PERF_SCALAR'], ascending=[False, False, False])
published_ranking['Rank'] = published_ranking.groupby(['GENDER', 'MAPPED_EVENT']).cumcount() + 1

published_ranking.to_csv('published_ranking_prod.csv', encoding='utf-8')

In [109]:
# Rank everyone for octc selection

all_ranking = final_df.sort_values(['MAPPED_EVENT','GENDER','PERF_SCALAR'], ascending=[False, False, False])
all_ranking['Rank'] = all_ranking.groupby(['GENDER', 'MAPPED_EVENT', 'TIER']).cumcount() + 1


In [565]:
# Convert time format for marathon and 5000m into mm:ss.00
# Choose the correct column indices or you will get erratic timings

import datetime

#s=247.779

#datetime.datetime.fromtimestamp(s).strftime('%M:%S.%f')

all_ranking=all_ranking.reset_index(drop=True)


#all_ranking[['2%', '3.5%', '5%']] = df[['2%', '3.5%', '5%']].apply(pd.to_numeric)


#all_ranking['2%'] = all_ranking['2%'].astype("string")
#all_ranking['3.5%'] = all_ranking['3.5%'].astype("string")
#all_ranking['5%'] = all_ranking['5%'].astype("string")


for i in range(len(all_ranking)):
        
    rowIndex = all_ranking.index[i]

    event=all_ranking.iloc[rowIndex,21]
        
    
    time_base2=all_ranking.iloc[rowIndex,25]
    time_base3=all_ranking.iloc[rowIndex,26]
    time_base5=all_ranking.iloc[rowIndex,27]
    
        
    if metric==None:
        continue
        
    if event=='800m' or event=='10,000m' or event=='5000m' or event=='3000m Steeplechase' or event=='1500m':
        
      #  print(i, event, time_base2, time_base3, time_base5)

        
        

            
        
        date_preconvert2 = datetime.datetime.utcfromtimestamp(time_base2)
        date_preconvert3 = datetime.datetime.utcfromtimestamp(time_base3)
        date_preconvert5 = datetime.datetime.utcfromtimestamp(time_base5)
        
    #    print(date_preconvert2, date_preconvert3, date_preconvert5)
            
        
        output2 = datetime.datetime.strftime(date_preconvert2, "%M:%S.%f")
        output3 = datetime.datetime.strftime(date_preconvert3, "%M:%S.%f")
        output5 = datetime.datetime.strftime(date_preconvert5, "%M:%S.%f")
            
     #   print(event, output2, output3, output5)

                    
       #     top_performers_clean.loc[rowIndex, '2%_timing'] = output2
       #     top_performers_clean.loc[rowIndex, '3.5%_timing'] = output3
       #     top_performers_clean.loc[rowIndex, '5%_timing'] = output5
            
   
        all_ranking.at[rowIndex, '2%'] = output2 # copy over time format
        all_ranking.at[rowIndex, '3.5%'] = output3
        all_ranking.at[rowIndex, '5%'] = output5


            


        
    elif event=='Marathon':
        
      #  print(time_base2, time_base3, time_base5)

        
        try:
            

        
            date_preconvert2 = datetime.datetime.utcfromtimestamp(time_base2)
            date_preconvert3 = datetime.datetime.utcfromtimestamp(time_base3)
            date_preconvert5 = datetime.datetime.utcfromtimestamp(time_base5)

            
            
            output2 = datetime.datetime.strftime(date_preconvert2, "%H:%M:%S")
            output3 = datetime.datetime.strftime(date_preconvert3, "%H:%M:%S")
            output5 = datetime.datetime.strftime(date_preconvert5, "%H:%M:%S")

            
        
        #    top_performers_clean.loc[rowIndex, '2%_timing'] = output2
        #    top_performers_clean.loc[rowIndex, '3.5%_timing'] = output3
        #    top_performers_clean.loc[rowIndex, '5%_timing'] = output5
            
            all_ranking.at[rowIndex, '2%'] = output2 # copy over time format
            all_ranking.at[rowIndex, '3.5%'] = output3
            all_ranking.at[rowIndex, '5%'] = output5

            
         #   print('output', output2, output3, output5)


        
        except:
            
            pass
                        
             
